# Urban Heat Islands in Indian Metrocities

## Export UHI / LST / PM2.5 tensors (monthly)

This cell downloads monthly PM2.5, MODIS LST (day/night), and several UHII collections for a 40×40 km region around each city center (40×40 @ 1 km). It also downloads annual 30 m land cover for each year. Outputs are saved per-city as PyTorch tensors with simple availability masks, plus a `meta.json` describing shapes and band order.

**Key outputs per city**
- `pm25.pt`        — (T=216, 1, 40, 40)
- `pm25_mask.pt`   — (216, 1, 40, 40)
- `lst.pt`         — (216, 2, 40, 40)  # Day, Night
- `lst_mask.pt`    — (216, 2, 40, 40)
- `uhii.pt`        — (216, B=8, 40, 40)
- `uhii_mask.pt`   — (216, B=8, 40, 40)
- `lc_annual.pt`   — (Y=18, 1, 1333, 1333)
- `meta.json`      — metadata, band order, file names

**Notes**
- Earth Engine credentials required; the cell attempts `ee.Authenticate()` if needed.
- Downloads are heavy — test with a single city or shorter year range first.
- Masks are simple (>0 or !=0); consider dataset-specific QA for production.



In [ ]:
import os, json
import torch
import numpy as np
import ee, geemap
from scipy.ndimage import zoom

cloud_project = 'refined-kite-414512'

try:
  ee.Initialize(project=cloud_project)
except Exception:
  ee.Authenticate()
  ee.Initialize(project=cloud_project)

CITY_CENTERS = {
    "Delhi":     (77.1025, 28.7041),
    "Mumbai":    (72.8777, 19.0760),
    "Kolkata":   (88.3639, 22.5726),
    "Chennai":   (80.2707, 13.0827),
    "Pune":      (73.8567, 18.5204),
    "Bengaluru": (77.5946, 12.9716),
}

YEARS  = list(range(2003, 2021))
MONTHS = list(range(1, 13))
T_STEPS = len(YEARS) * len(MONTHS)  # 216

UHII_COLLECTIONS = {
    # order matters; this becomes band axis order B
    "AMOD2": "projects/sat-io/open-datasets/UHII/AMOD2",  # SUHI (all-sky)
    "MOD1":  "projects/sat-io/open-datasets/UHII/MOD1",   # SUHI (Terra daily)
    "MOD2":  "projects/sat-io/open-datasets/UHII/MOD2",   # SUHI (Terra 8-day)
    "MYD1":  "projects/sat-io/open-datasets/UHII/MYD1",   # SUHI (Aqua daily)
    "MYD2":  "projects/sat-io/open-datasets/UHII/MYD2",   # SUHI (Aqua 8-day)
    "SAT":   "projects/sat-io/open-datasets/UHII/SAT",    # CUHI (air temp)
    "SMOD2": "projects/sat-io/open-datasets/UHII/SMOD2",  # SUHI seamless Terra
    "SMYD1": "projects/sat-io/open-datasets/UHII/SMYD1",  # SUHI seamless Aqua
}
UHII_BANDS = list(UHII_COLLECTIONS.keys())  # fixed order
B = len(UHII_BANDS)

LC_ANNUAL = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")

GRID_SIZE_KM = 40
PIXEL_SIZE_1KM = 1000
PIXEL_SIZE_30M = 30

WIDTH_PX_1KM = 40
WIDTH_PX_30M = 1333

def create_region_geometry(lon: float, lat: float, size_km: float) -> ee.Geometry:
    deg_per_km_lon = 1.0 / (111.32 * np.cos(np.radians(lat)))
    deg_per_km_lat = 1.0 / 110.54
    half_lon = (size_km / 2) * deg_per_km_lon
    half_lat = (size_km / 2) * deg_per_km_lat
    return ee.Geometry.Rectangle([lon - half_lon, lat - half_lat, lon + half_lon, lat + half_lat])

CITY_REGIONS = {name: create_region_geometry(*coords, GRID_SIZE_KM) for name, coords in CITY_CENTERS.items()}

def resize_array_to_target(arr: np.ndarray, target_shape: tuple) -> np.ndarray:
    """Resize array to target shape using scipy zoom."""
    if arr.shape == target_shape:
        return arr

    if len(arr.shape) == 2:  # (H, W)
        zoom_factors = (target_shape[0] / arr.shape[0], target_shape[1] / arr.shape[1])
        return zoom(arr, zoom_factors, order=1, mode='nearest')
    elif len(arr.shape) == 3:  # (C, H, W)
        zoom_factors = (1, target_shape[1] / arr.shape[1], target_shape[2] / arr.shape[2])
        return zoom(arr, zoom_factors, order=1, mode='nearest')
    else:
        return arr

def ee_to_numpy_singleband(ee_img: ee.Image, region: ee.Geometry, scale: int, target_shape=(40, 40)) -> np.ndarray:
    """Download single-band image → (H,W) float32 with consistent shape."""
    try:
        arr = geemap.ee_to_numpy(ee_img, region=region, scale=scale)
        if arr is None:
            return np.zeros(target_shape, np.float32)

        # geemap returns (H,W,1)
        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = arr[..., 0]

        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

        # Resize to target shape if needed
        arr = resize_array_to_target(arr, target_shape)

        return arr
    except Exception as e:
        print(f"    Error in ee_to_numpy_singleband: {e}")
        return np.zeros(target_shape, np.float32)

def ee_to_numpy_multiband(ee_img: ee.Image, band_names, region: ee.Geometry, scale: int, target_shape=(2, 40, 40)) -> np.ndarray:
    """Download multi-band image → (C,H,W) float32 in the given band order with consistent shape."""
    try:
        img = ee_img.select(band_names)
        arr = geemap.ee_to_numpy(img, region=region, scale=scale)
        if arr is None:
            return np.zeros(target_shape, np.float32)

        # shape from geemap: (H,W,C); we want (C,H,W)
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
        if arr.ndim == 2:  # (H,W) → add channel dim
            arr = arr[:, :, None]
        arr = np.transpose(arr, (2, 0, 1))  # (H,W,C) → (C,H,W)

        # Resize to target shape if needed
        arr = resize_array_to_target(arr, target_shape)

        return arr
    except Exception as e:
        print(f"    Error in ee_to_numpy_multiband: {e}")
        return np.zeros(target_shape, np.float32)

def to1km(img, region):
    return img.reproject(img.projection().atScale(PIXEL_SIZE_1KM)).clip(region)

def lc_band_for_year(year: int, region: ee.Geometry) -> ee.Image:
    mosaic = LC_ANNUAL.filterBounds(region).mosaic()
    band = f"b{year - 1999}"
    return mosaic.select([band]).rename("LC30m_raw").clip(region)

def recode_classes(img: ee.Image) -> ee.Image:
    # (Keep your recode mapping; you can adjust src/dst if you refine groupings)
    src = [0, 10,11,12,20,51,52,61,62,71,72,81,82,91,92,
           120,121,122,130,140,150,152,153,181,182,183,184,185,186,187,
           190,200,201,202,210,220,250]
    dst = [0] + list(range(1, len(src)-1)) + [0]
    return img.remap(src, dst).toInt16().rename("LC30m_recode")

def fetch_lst_month(year: int, month: int, region: ee.Geometry) -> ee.Image:
    start = ee.Date.fromYMD(year, month, 1)
    end   = start.advance(1, 'month')
    coll = (ee.ImageCollection("MODIS/061/MOD11A2")
            .filterDate(start, end)
            .filterBounds(region)
            .select(["LST_Day_1km", "LST_Night_1km"]))
    img = coll.mean().multiply(0.02).subtract(273.15)  # K→°C
    img = img.rename(["LST_Day", "LST_Night"])
    return to1km(img, region)

def availability_mask_from_array(arr: np.ndarray, thresh=0.0) -> np.ndarray:
    """1 where valid; here we treat > thresh as valid (suitable for UHII where <=0 masked earlier)."""
    return (arr > thresh).astype(np.float32)

def build_and_save_tensors(city_name: str, region: ee.Geometry):
    print(f"\nProcessing {city_name}…")
    # Containers
    lst_list, lst_mask_list = [], []
    pm_list, pm_mask_list   = [], []
    uhii_list, uhii_mask_list = [], []

    # ---------- PM2.5 ----------
    print("  PM2.5 monthly…")
    for yr in YEARS:
        for mo in MONTHS:
            start = ee.Date.fromYMD(yr, mo, 1)
            end   = start.advance(1, 'month')
            try:
                pm_img = (ee.ImageCollection("projects/sat-io/open-datasets/GLOBAL-SATELLITE-PM25/MONTHLY")
                          .filterDate(start, end)
                          .filterBounds(region)
                          .mean())
                pm_img = to1km(pm_img.rename(["PM25"]), region)
                pm_arr = ee_to_numpy_singleband(pm_img, region, PIXEL_SIZE_1KM, (WIDTH_PX_1KM, WIDTH_PX_1KM))  # (H,W)
            except Exception as e:
                print(f"    PM2.5 {yr}-{mo:02d} error: {e}")
                pm_arr = np.zeros((WIDTH_PX_1KM, WIDTH_PX_1KM), np.float32)

            pm_mask = availability_mask_from_array(pm_arr, thresh=0.0)  # >0 considered valid
            pm_list.append(torch.from_numpy(pm_arr)[None, ...])         # (1,H,W)
            pm_mask_list.append(torch.from_numpy(pm_mask)[None, ...])   # (1,H,W)

    pm_tensor      = torch.stack(pm_list)      # (T,1,H,W)
    pm_mask_tensor = torch.stack(pm_mask_list) # (T,1,H,W)

    # ---------- LST (Day/Night) ----------
    print("  LST monthly (Day & Night)…")
    for yr in YEARS:
        for mo in MONTHS:
            try:
                lst_img = fetch_lst_month(yr, mo, region)   # two bands
                lst_arr = ee_to_numpy_multiband(lst_img, ["LST_Day", "LST_Night"], region, PIXEL_SIZE_1KM, (2, WIDTH_PX_1KM, WIDTH_PX_1KM))  # (2,H,W)
            except Exception as e:
                print(f"    LST {yr}-{mo:02d} error: {e}")
                lst_arr = np.zeros((2, WIDTH_PX_1KM, WIDTH_PX_1KM), np.float32)

            # Valid if not 0 (you may prefer a more nuanced mask; e.g., nonzero & non-NaN)
            lst_mask = (lst_arr != 0.0).astype(np.float32)

            lst_list.append(torch.from_numpy(lst_arr))         # (2,H,W)
            lst_mask_list.append(torch.from_numpy(lst_mask))   # (2,H,W)

    lst_tensor      = torch.stack(lst_list)        # (T,2,H,W)
    lst_mask_tensor = torch.stack(lst_mask_list)   # (T,2,H,W)

    # ---------- UHII (per collection) ----------
    print("  UHII monthly (per collection)…")
    for yr in YEARS:
        for mo in MONTHS:
            band_preds = []
            band_masks = []
            for band_name, asset in UHII_COLLECTIONS.items():
                try:
                    coll = (ee.ImageCollection(asset)
                            .filter(ee.Filter.calendarRange(yr, yr, 'year'))
                            .filter(ee.Filter.calendarRange(mo, mo, 'month'))
                            .filterBounds(region))
                    if coll.size().getInfo() > 0:
                        # mean over that month for this band; scale to °C if needed
                        img = coll.mean().updateMask(coll.mean().gt(0))  # >0 valid (as you did)
                        img = to1km(img.rename([band_name]), region).multiply(0.01)
                        arr = ee_to_numpy_singleband(img, region, PIXEL_SIZE_1KM, (WIDTH_PX_1KM, WIDTH_PX_1KM))  # (H,W)
                        msk = availability_mask_from_array(arr, thresh=0.0)
                    else:
                        arr = np.zeros((WIDTH_PX_1KM, WIDTH_PX_1KM), np.float32)
                        msk = np.zeros_like(arr, np.float32)
                except Exception as e:
                    print(f"    UHII {band_name} {yr}-{mo:02d} error: {e}")
                    arr = np.zeros((WIDTH_PX_1KM, WIDTH_PX_1KM), np.float32)
                    msk = np.zeros_like(arr, np.float32)

                band_preds.append(torch.from_numpy(arr)[None, ...])  # (1,H,W)
                band_masks.append(torch.from_numpy(msk)[None, ...])  # (1,H,W)

            uhii_list.append(torch.cat(band_preds, dim=0))  # (B,H,W)
            uhii_mask_list.append(torch.cat(band_masks, dim=0))  # (B,H,W)

    uhii_tensor      = torch.stack(uhii_list)        # (T,B,H,W)
    uhii_mask_tensor = torch.stack(uhii_mask_list)   # (T,B,H,W)

    # ---------- Land cover (annual 30m) ----------
    print("  Land cover annual (30 m)…")
    lc_annual_tiles = []
    for yr in YEARS:
        try:
            lc_img = recode_classes(lc_band_for_year(yr, region))
            arr = geemap.ee_to_numpy(lc_img, region=region, scale=PIXEL_SIZE_30M)
            if arr is None:
                arr = np.zeros((WIDTH_PX_30M, WIDTH_PX_30M), np.int16)
            else:
                if arr.ndim == 3 and arr.shape[-1] == 1:
                    arr = arr[..., 0]
                arr = np.nan_to_num(arr, nan=0.0).astype(np.int16)
                # Resize to target shape if needed
                arr = resize_array_to_target(arr, (WIDTH_PX_30M, WIDTH_PX_30M)).astype(np.int16)
        except Exception as e:
            print(f"    LC {yr} error: {e}")
            arr = np.zeros((WIDTH_PX_30M, WIDTH_PX_30M), np.int16)
        lc_annual_tiles.append(torch.from_numpy(arr)[None, ...])  # (1,1333,1333)

    lc_annual_tensor = torch.stack(lc_annual_tiles)  # (Y,1,1333,1333)

    # ---------- Save ----------
    out_dir = f"/content/UHI_Tensors/{city_name}"
    os.makedirs(out_dir, exist_ok=True)

    torch.save(pm_tensor,            f"{out_dir}/pm25.pt")
    torch.save(pm_mask_tensor,       f"{out_dir}/pm25_mask.pt")
    torch.save(lst_tensor,           f"{out_dir}/lst.pt")
    torch.save(lst_mask_tensor,      f"{out_dir}/lst_mask.pt")
    torch.save(uhii_tensor,          f"{out_dir}/uhii.pt")
    torch.save(uhii_mask_tensor,     f"{out_dir}/uhii_mask.pt")
    torch.save(lc_annual_tensor,     f"{out_dir}/lc_annual.pt")

    meta = {
        "years": YEARS,
        "months": MONTHS,
        "t_steps": T_STEPS,
        "grid_1km": [WIDTH_PX_1KM, WIDTH_PX_1KM],
        "grid_30m": [WIDTH_PX_30M, WIDTH_PX_30M],
        "uhii_bands_order": UHII_BANDS,
        "lst_channels": ["LST_Day", "LST_Night"],
        "pm25_channel": ["PM25"],
        "files": {
            "pm25": "pm25.pt",
            "pm25_mask": "pm25_mask.pt",
            "lst": "lst.pt",
            "lst_mask": "lst_mask.pt",
            "uhii": "uhii.pt",
            "uhii_mask": "uhii_mask.pt",
            "lc_annual": "lc_annual.pt"
        }
    }
    with open(f"{out_dir}/meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    print(f"  ✓ Saved tensors for {city_name}")
    print(f"    PM2.5:      {tuple(pm_tensor.shape)}")
    print(f"    LST:        {tuple(lst_tensor.shape)}")
    print(f"    UHII:       {tuple(uhii_tensor.shape)} | mask {tuple(uhii_mask_tensor.shape)}")
    print(f"    LC annual:  {tuple(lc_annual_tensor.shape)}")

# -------------- main --------------
print("Starting band-aware, masked tensor export…")
os.makedirs("/content/UHI_Tensors", exist_ok=True)
for city, region in CITY_REGIONS.items():
    try:
        build_and_save_tensors(city, region)
    except Exception as e:
        print(f"Error processing {city}: {e}")
print("Done.")

KeyboardInterrupt: 

In [ ]:
# Store the tensors downloaded (Colab used on my end), thus storing to zip file for future usage.

import os
import zipfile
from google.colab import files

def create_zip(folder_path="/content/UHI_Tensors", zip_name="UHI_Tensors.zip"):
    """
    Create a ZIP file of the UHI tensors folder and download it to local PC
    """
    zip_path = f"/content/{zip_name}"

    print(f"Creating ZIP archive: {zip_name}")
    print(f"Source folder: {folder_path}")

    # Create ZIP file
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Walk through all files in the folder
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Get relative path for archive
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)
                print(f"  Added: {arcname}")

create_zip()

Creating ZIP archive: UHI_Tensors.zip
Source folder: /content/UHI_Tensors
  Added: Bengaluru/uhii_mask.pt
  Added: Bengaluru/pm25.pt
  Added: Bengaluru/lc_annual.pt
  Added: Bengaluru/uhii.pt
  Added: Bengaluru/meta.json
  Added: Bengaluru/lst_mask.pt
  Added: Bengaluru/lst.pt
  Added: Bengaluru/pm25_mask.pt
  Added: Delhi/uhii_mask.pt
  Added: Delhi/pm25.pt
  Added: Delhi/lc_annual.pt
  Added: Delhi/uhii.pt
  Added: Delhi/meta.json
  Added: Delhi/lst_mask.pt
  Added: Delhi/lst.pt
  Added: Delhi/pm25_mask.pt
  Added: Pune/uhii_mask.pt
  Added: Pune/pm25.pt
  Added: Pune/lc_annual.pt
  Added: Pune/uhii.pt
  Added: Pune/meta.json
  Added: Pune/lst_mask.pt
  Added: Pune/lst.pt
  Added: Pune/pm25_mask.pt
  Added: Mumbai/uhii_mask.pt
  Added: Mumbai/pm25.pt
  Added: Mumbai/lc_annual.pt
  Added: Mumbai/uhii.pt
  Added: Mumbai/meta.json
  Added: Mumbai/lst_mask.pt
  Added: Mumbai/lst.pt
  Added: Mumbai/pm25_mask.pt
  Added: Kolkata/uhii_mask.pt
  Added: Kolkata/pm25.pt
  Added: Kolkata/lc_ann

AttributeError: 'list' object has no attribute 'download'

## Finalize & prepare tensors (expand annual LC → monthly)

This cell finalizes the exported city tensors: validates shapes, constructs a month→year index, replicates annual 30 m land-cover into a monthly series aligned with the monthly time axis T, saves `lc.pt` and `month_to_year.pt`, and updates `meta.json`.

**Notes**
- Replicating 18 annual 1333×1333 tiles to monthly (T=216) is memory-heavy (~732 MiB if `int16`, ~1.46 GiB if `float32`).
- If you have limited RAM/disk, keep the annual LC and use `month_to_year` at runtime instead of replication.
- Fix a small bug before running: replace `raise ValueValue(...)` with `raise ValueError(...)` in `_validate_shape`.


In [1]:
# %%colab
import os, json, zipfile, sys
from typing import Dict, List, Tuple, Optional
import torch
import numpy as np

# ==============================
# CONFIG — edit if needed
# ==============================
ZIP_PATH   = "./UHI_Tensors.zip"   # set to "" if you already have a folder
ROOT_DIR   = "./"       # where city subfolders live (if ZIP is "", we use this)
EXPECT_CITIES = {"Delhi","Mumbai","Kolkata","Chennai","Pune","Bengaluru"}  # just to sanity-check
FORCE_FLOAT32 = True   # ensure float32 for data, float32 {0,1} for masks
VERBOSE        = True  # print shapes

# ==============================
# HELPERS
# ==============================
def extract_zip_if_needed(zip_path: str, out_dir: str) -> str:
    if not zip_path or not os.path.exists(zip_path):
        return out_dir
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(os.path.dirname(out_dir))
    # try to find the folder that contains our cities
    if os.path.isdir(out_dir):
        return out_dir
    # fallback: find a folder that contains several expected cities
    candidate_root = None
    for root, dirs, files in os.walk(os.path.dirname(out_dir)):
        hits = [d for d in dirs if d in EXPECT_CITIES]
        if len(hits) >= 3:
            candidate_root = root
            break
    return candidate_root or out_dir

def _as_float32(x: torch.Tensor) -> torch.Tensor:
    return x.float().contiguous()

def _mask01(x: torch.Tensor) -> torch.Tensor:
    # Convert any nonzero -> 1, keep zeros -> 0, cast to float32
    return (x != 0).float().contiguous()

def _validate_shape(name: str, t: torch.Tensor, exp: Tuple[int, ...]):
    if tuple(t.shape) != tuple(exp):
        raise ValueError(f"{name} has shape {tuple(t.shape)} but expected {tuple(exp)}")

def _load_meta(city_dir: str) -> Dict:
    p = os.path.join(city_dir, "meta.json")
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing meta.json in {city_dir}")
    with open(p, "r") as f:
        return json.load(f)

def _save_meta(city_dir: str, meta: Dict):
    with open(os.path.join(city_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

# ==============================
# MAIN “FINALIZATION” LOGIC
# ==============================
root = extract_zip_if_needed(ZIP_PATH, ROOT_DIR)
if not os.path.isdir(root):
    raise RuntimeError(f"Could not find root folder with city subfolders at: {root}")

city_dirs = [os.path.join(root, d) for d in sorted(os.listdir(root))
             if os.path.isdir(os.path.join(root, d))]
if VERBOSE:
    print(f"Found {len(city_dirs)} city folders under {root}: {[os.path.basename(d) for d in city_dirs]}")

for city_dir in city_dirs:
    city = os.path.basename(city_dir)
    if city not in EXPECT_CITIES:
        # skip unknown folders silently, or warn:
        if VERBOSE:
            print(f"Skipping non-city folder: {city_dir}")
        continue

    print(f"\n=== Finalizing tensors for {city} ===")
    meta = _load_meta(city_dir)

    # Expected grids and steps from meta
    H1, W1 = meta.get("grid_1km", [40, 40])
    H30, W30 = meta.get("grid_30m", [1333, 1333])
    YEARS   = meta.get("years", [])
    MONTHS  = meta.get("months", [])
    T       = meta.get("t_steps", len(YEARS) * len(MONTHS))
    uhii_bands_order = meta.get("uhii_bands_order", [])
    B = len(uhii_bands_order)

    files = meta.get("files", {})

    # -------- Load tensors --------
    def _load(name, expect_shape=None, make_float=False, make_mask=False):
        # path via meta["files"][name] if present; else fallback to name.pt
        p = os.path.join(city_dir, files.get(name, f"{name}.pt"))
        if not os.path.exists(p):
            p = os.path.join(city_dir, f"{name}.pt")
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing {name}.pt in {city_dir}")
        t = torch.load(p, map_location="cpu")
        if make_mask:
            t = _mask01(t)
        elif make_float:
            t = _as_float32(t)
        if expect_shape is not None:
            _validate_shape(name, t, expect_shape)
        return t

    pm25       = _load("pm25",       (T, 1, H1, W1), make_float=FORCE_FLOAT32)
    pm25_mask  = _load("pm25_mask",  (T, 1, H1, W1), make_mask=True)
    lst        = _load("lst",        (T, 2, H1, W1), make_float=FORCE_FLOAT32)
    lst_mask   = _load("lst_mask",   (T, 2, H1, W1), make_mask=True)
    uhii       = _load("uhii",       (T, B, H1, W1), make_float=FORCE_FLOAT32)
    uhii_mask  = _load("uhii_mask",  (T, B, H1, W1), make_mask=True)
    lc_annual  = _load("lc_annual",  None,            make_float=False)  # keep int16

    # Validate lc_annual rank & basics
    if lc_annual.ndim != 4 or lc_annual.shape[1:] != (1, H30, W30):
        raise ValueError(f"lc_annual has shape {tuple(lc_annual.shape)}; expected (Y, 1, {H30}, {W30})")
    Y = lc_annual.shape[0]

    # -------- Build month→year index & lc monthly replication --------
    # If YEARS was exported as range(2003..2020) length Y, we map each month t to year_index = t // 12.
    # If YEARS is missing, we still build an index using Y.
    if not YEARS:
        YEARS = list(range(Y))  # dummy indices
    if not MONTHS:
        MONTHS = list(range(1, 13))
    if len(YEARS) != Y:
        print(f"Warning: meta.years has {len(YEARS)} items but lc_annual has {Y}; using Y from lc_annual.")

    # month_to_year: (T,) tensor with values in [0..Y-1]
    month_to_year = torch.tensor([min(Y-1, t // 12) for t in range(T)], dtype=torch.long)
    # Replicate lc_annual -> monthly lc: (T, 1, 1333, 1333)
    # Efficient way: repeat_interleave along time with repeats=12, then trim to T
    lc_monthly = lc_annual.repeat_interleave(12, dim=0)[:T].contiguous()  # (Y*12,1,H30,W30) → (T,1,H30,W30)

    # -------- Sanity checks (light) --------
    _validate_shape("pm25",      pm25,      (T, 1, H1, W1))
    _validate_shape("pm25_mask", pm25_mask, (T, 1, H1, W1))
    _validate_shape("lst",       lst,       (T, 2, H1, W1))
    _validate_shape("lst_mask",  lst_mask,  (T, 2, H1, W1))
    _validate_shape("uhii",      uhii,      (T, B, H1, W1))
    _validate_shape("uhii_mask", uhii_mask, (T, B, H1, W1))
    _validate_shape("lc_monthly", lc_monthly, (T, 1, H30, W30))

    # -------- Save new tensors --------
    lc_monthly_path   = os.path.join(city_dir, "lc.pt")
    m2y_path          = os.path.join(city_dir, "month_to_year.pt")

    torch.save(lc_monthly, lc_monthly_path)
    torch.save(month_to_year, m2y_path)

    # Update meta.json with new files + mapping info
    files["lc"] = "lc.pt"
    meta["files"] = files
    meta["month_to_year_note"] = "month_to_year[t] gives the year slice index in lc_annual corresponding to month t"
    meta["lc_monthly_shape"] = [int(x) for x in lc_monthly.shape]
    meta["month_to_year_file"] = "month_to_year.pt"
    _save_meta(city_dir, meta)

    # -------- Print shapes --------
    if VERBOSE:
        print(f"  pm25:       {tuple(pm25.shape)}   | mask {tuple(pm25_mask.shape)}")
        print(f"  lst:        {tuple(lst.shape)}    | mask {tuple(lst_mask.shape)}")
        print(f"  uhii:       {tuple(uhii.shape)}   | mask {tuple(uhii_mask.shape)}")
        print(f"  lc_annual:  {tuple(lc_annual.shape)}")
        print(f"  lc (NEW):   {tuple(lc_monthly.shape)}")
        print(f"  month_to_year: {tuple(month_to_year.shape)}  (first 12 = {month_to_year[:12].tolist()})")

print("\nAll cities finalized. You can now feed (pm25, lst, lc) with their masks and uhii targets into the model.")


Found 9 city folders under ./: ['.venv', 'Bengaluru', 'Chennai', 'Delhi', 'Kolkata', 'Mumbai', 'Pune', 'evaluation_results', 'runs']
Skipping non-city folder: ./.venv

=== Finalizing tensors for Bengaluru ===
  pm25:       (216, 1, 40, 40)   | mask (216, 1, 40, 40)
  lst:        (216, 2, 40, 40)    | mask (216, 2, 40, 40)
  uhii:       (216, 8, 40, 40)   | mask (216, 8, 40, 40)
  lc_annual:  (18, 1, 1333, 1333)
  lc (NEW):   (216, 1, 1333, 1333)
  month_to_year: (216,)  (first 12 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

=== Finalizing tensors for Chennai ===
  pm25:       (216, 1, 40, 40)   | mask (216, 1, 40, 40)
  lst:        (216, 2, 40, 40)    | mask (216, 2, 40, 40)
  uhii:       (216, 8, 40, 40)   | mask (216, 8, 40, 40)
  lc_annual:  (18, 1, 1333, 1333)
  lc (NEW):   (216, 1, 1333, 1333)
  month_to_year: (216,)  (first 12 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

=== Finalizing tensors for Delhi ===
  pm25:       (216, 1, 40, 40)   | mask (216, 1, 40, 40)
  lst:        (216, 2, 4

## Interactive multi-city tensor viewer (band-aware, masks, monthly LC)

This cell provides an interactive visualization panel for per-city tensors:
- UHII (multi-band), PM2.5, MODIS LST (day/night), and 30 m Land Cover.
- Select year/month, UHII band or LST channel, choose cities (multi-select), and optionally overlay masks to highlight invalid pixels.
- Land cover prefers a monthly `lc.pt` but will fall back to `lc_annual` + `month_to_year.pt` if available.


In [51]:
# =========================
# Interactive multi-city tensor viewer (band-aware, masks, monthly LC)
# =========================
import os, json
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from ipywidgets import widgets, HBox, VBox, interactive_output, SelectMultiple
from IPython.display import display, clear_output

# ---------- 0) Config ----------
ROOT_DIR = "./"  # Local folder in same directory as notebook
BASE_YEAR = 2003
EXPECTED_CITIES = {"Delhi","Mumbai","Kolkata","Chennai","Pune","Bengaluru"}

# ---------- 1) Load city packs ----------
def load_city_packs(root_dir):
    city_data = {}
    if not os.path.isdir(root_dir):
        raise RuntimeError(f"Folder not found: {root_dir}")
    for d in sorted(os.listdir(root_dir)):
        cdir = os.path.join(root_dir, d)
        if not os.path.isdir(cdir) or d not in EXPECTED_CITIES:
            continue
        meta_path = os.path.join(cdir, "meta.json")
        if not os.path.exists(meta_path):
            print(f"Skipping {d}: meta.json missing")
            continue
        with open(meta_path, "r") as f:
            meta = json.load(f)

        files = meta.get("files", {})
        def _p(name, default=None):
            fname = files.get(name, f"{name}.pt")
            path = os.path.join(cdir, fname if default is None else default)
            if not os.path.exists(path) and default is None:
                path = os.path.join(cdir, f"{name}.pt")
            return path

        # Load tensors (map_location=cpu keeps it light)
        tensors = {}
        def _load_or_none(k):
            p = _p(k)
            return torch.load(p, map_location="cpu") if os.path.exists(p) else None

        tensors["uhii"]      = _load_or_none("uhii")        # (T,B,40,40)
        tensors["uhii_mask"] = _load_or_none("uhii_mask")   # (T,B,40,40)
        tensors["lst"]       = _load_or_none("lst")         # (T,2,40,40)
        tensors["lst_mask"]  = _load_or_none("lst_mask")    # (T,2,40,40)
        tensors["pm25"]      = _load_or_none("pm25")        # (T,1,40,40)
        tensors["pm25_mask"] = _load_or_none("pm25_mask")   # (T,1,40,40)
        # Prefer monthly LC 'lc.pt'; fallback to annual (we'll expand on the fly)
        lc_monthly_path = _p("lc", default="lc.pt")
        if os.path.exists(lc_monthly_path):
            tensors["lc"] = torch.load(lc_monthly_path, map_location="cpu")  # (T,1,1333,1333)
        else:
            # fallback: expand lc_annual -> monthly
            lc_annual = _load_or_none("lc_annual")  # (Y,1,1333,1333)
            month_to_year_p = os.path.join(cdir, "month_to_year.pt")
            if lc_annual is not None and os.path.exists(month_to_year_p):
                m2y = torch.load(month_to_year_p, map_location="cpu").long()
                tensors["lc"] = lc_annual[m2y].contiguous()
            else:
                tensors["lc"] = None

        # Record
        tensors["meta"] = meta
        city_data[d] = tensors
    if not city_data:
        raise RuntimeError(f"No city folders with tensors found under {root_dir}")
    return city_data

city_data = load_city_packs(ROOT_DIR)

# Common shapes/metadata sanity
first_city = next(iter(city_data))
T = city_data[first_city]["pm25"].shape[0]
H1, W1 = city_data[first_city]["pm25"].shape[-2:]
H30, W30 = city_data[first_city]["lc"].shape[-2:]

# UHII band names (assumed same order across cities)
UHII_BANDS = city_data[first_city]["meta"].get("uhii_bands_order", [])
if not UHII_BANDS:
    UHII_BANDS = [f"band_{i}" for i in range(city_data[first_city]["uhii"].shape[1])]
UHII_BAND_OPTIONS = [(name, i) for i, name in enumerate(UHII_BANDS)]

# ---------- 2) Colormaps & styling ----------
UHII_VMIN, UHII_VMAX = 0.1, 2.5
UHII_CMAP = "inferno"
PM25_CMAP = "viridis"

_lst_colors = ['#313695', '#4575b4', '#74add1', '#abd9e9', '#e0f3f8',
               '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026']
LST_CMAP = mcolors.LinearSegmentedColormap.from_list('custom', _lst_colors, N=51)
LST_NORM = mcolors.BoundaryNorm(np.arange(0, 52, 1), ncolors=51)

# 36-class palette for recoded LC
_lc_palette = [
  "#ffff64", "#ffff64", "#ffff00", "#aaf0f0", "#4c7300", "#006400", "#a8c800", "#00a000",
  "#005000", "#003c00", "#286400", "#285000", "#a0b432", "#788200", "#966400", "#964b00",
  "#966400", "#ffb432", "#ffdcd2", "#ffebaf", "#ffd278", "#ffebaf", "#00a884", "#73ffdf",
  "#9ebb3b", "#828282", "#f57ab6", "#66cdab", "#444f89", "#c31400", "#fff5d7", "#dcdcdc",
  "#fff5d7", "#0046c8", "#ffffff", "#ffffff"
]
LC_CMAP  = mcolors.ListedColormap(_lc_palette)
LC_NORM  = mcolors.BoundaryNorm(np.arange(1, 38), ncolors=36)

MONTHS = [("January",1),("February",2),("March",3),("April",4),("May",5),("June",6),
          ("July",7),("August",8),("September",9),("October",10),("November",11),("December",12)]

def idx_from_year_month(year:int, month:int, base_year:int=BASE_YEAR) -> int:
    return (year - base_year) * 12 + (month - 1)

def robust_minmax_across(arrays, low=2, high=98):
    flat = []
    for a in arrays:
        a = np.asarray(a)
        if a.size == 0: continue
        a = a[np.isfinite(a)]
        if a.size: flat.append(a)
    if not flat: return None
    flat = np.concatenate(flat)
    return (np.nanpercentile(flat, low), np.nanpercentile(flat, high))

# ---------- 3) UI widgets ----------
dataset_labels = {
    "uhii": "Urban Heat Island (UHII)",
    "pm25": "PM2.5",
    "lst":  "Land Surface Temperature (°C)",
    "lc":   "Land Cover (30 m)"
}
dataset_dd = widgets.Dropdown(
    options=[(v,k) for k,v in dataset_labels.items()],
    value="uhii",
    description="Dataset:",
    layout=widgets.Layout(width="280px")
)
year_dd = widgets.Dropdown(
    options=list(range(BASE_YEAR, BASE_YEAR + T//12)),
    value=2010,
    description="Year:",
    layout=widgets.Layout(width="140px")
)
month_dd = widgets.Dropdown(
    options=MONTHS,
    value=1,
    description="Month:",
    layout=widgets.Layout(width="200px")
)
uhii_band_dd = widgets.Dropdown(
    options=UHII_BAND_OPTIONS,
    value=UHII_BAND_OPTIONS[0][1] if UHII_BAND_OPTIONS else 0,
    description="UHII band:",
    layout=widgets.Layout(width="360px")
)
lst_channel_dd = widgets.Dropdown(
    options=[("Day",0), ("Night",1)],
    value=0,
    description="LST ch:",
    layout=widgets.Layout(width="180px")
)
show_mask_cb = widgets.Checkbox(
    value=False, description="Overlay mask", indent=False, layout=widgets.Layout(width="150px")
)

# City selector (multi-select)
city_names_sorted = sorted([c for c in city_data.keys()], key=lambda x: ["Mumbai","Bengaluru","Chennai","Pune","Delhi","Kolkata","Other"].index(x) if x in ["Mumbai","Bengaluru","Chennai","Pune","Delhi","Kolkata"] else 999)
city_sel = SelectMultiple(
    options=city_names_sorted,
    value=tuple(city_names_sorted),  # default all
    description="Cities:",
    layout=widgets.Layout(width="260px", height="120px")
)

controls_row1 = HBox([dataset_dd, year_dd, month_dd, show_mask_cb])
controls_row2 = HBox([uhii_band_dd, lst_channel_dd, city_sel])

# ---------- 4) Renderer ----------
def render_grid(dataset_key:str, year:int, month:int, uhii_band:int, lst_channel:int, show_mask, cities:tuple):
    clear_output(wait=True)
    print(f"Rendering: {dataset_labels[dataset_key]} — {MONTHS[month-1][0]} {year}")
    t_idx = idx_from_year_month(year, month, BASE_YEAR)
    if t_idx < 0 or t_idx >= T:
        print("Time index out of range.")
        return

    # Choose city order
    selected_cities = list(cities) if cities else list(city_data.keys())
    preferred = ["Mumbai","Bengaluru","Chennai","Pune","Delhi","Kolkata"]
    selected_cities = sorted(selected_cities, key=lambda c: (preferred.index(c) if c in preferred else 999, c))

    # Collect frames + (optional) masks
    frames = []
    masks  = []
    for city in selected_cities:
        pack = city_data[city]
        if dataset_key == "uhii":
            if pack["uhii"] is None: continue
            arr = pack["uhii"][t_idx, uhii_band].cpu().numpy()
            msk = pack["uhii_mask"][t_idx, uhii_band].cpu().numpy() if pack["uhii_mask"] is not None else None
            frames.append((city, arr))
            masks.append((city, msk))
        elif dataset_key == "pm25":
            if pack["pm25"] is None: continue
            arr = pack["pm25"][t_idx, 0].cpu().numpy()
            msk = pack["pm25_mask"][t_idx, 0].cpu().numpy() if pack["pm25_mask"] is not None else None
            frames.append((city, arr)); masks.append((city, msk))
        elif dataset_key == "lst":
            if pack["lst"] is None: continue
            arr = pack["lst"][t_idx, lst_channel].cpu().numpy()
            msk = pack["lst_mask"][t_idx, lst_channel].cpu().numpy() if pack["lst_mask"] is not None else None
            frames.append((city, arr)); masks.append((city, msk))
        elif dataset_key == "lc":
            if pack["lc"] is None: continue
            arr = pack["lc"][t_idx, 0].cpu().numpy()  # (1333,1333)
            frames.append((city, arr)); masks.append((city, None))
    if not frames:
        print("No frames available for this selection.")
        return

    # Decide color scaling & labels
    vmin, vmax, cmap, norm, cbar_label = None, None, None, None, ""
    if dataset_key == "uhii":
        vmin, vmax = UHII_VMIN, UHII_VMAX
        cmap, norm = UHII_CMAP, None
        cbar_label = f"UHII (band: {UHII_BANDS[uhii_band] if UHII_BANDS else uhii_band})"
    elif dataset_key == "pm25":
        cmap, norm = PM25_CMAP, None
        vmin, vmax = robust_minmax_across([f[1] for f in frames], 2, 98) or (0.0, 150.0)
        cbar_label = "PM2.5 (µg/m³)"
    elif dataset_key == "lst":
        cmap, norm = LST_CMAP, LST_NORM
        cbar_label = f"LST (°C) — {'Day' if lst_channel==0 else 'Night'}"
    elif dataset_key == "lc":
        cmap, norm = LC_CMAP, LC_NORM
        cbar_label = "Land Cover (class index)"

    # Grid layout up to 6 per page (2x3); if more, still show 6 (edit if you want pagination)
    rows, cols = 2, 3
    N = min(len(frames), rows*cols)
    fig, axes = plt.subplots(rows, cols, figsize=(18, 10))
    axes = axes.ravel()

    for ax in axes:
        ax.axis("off")

    for i, (city, arr) in enumerate(frames[:N]):
        ax = axes[i]
        a = np.array(arr, dtype=float)
        a[~np.isfinite(a)] = np.nan

        if dataset_key in ("lc", "lst"):
            im = ax.imshow(a, cmap=cmap, norm=norm, interpolation="nearest")
        else:
            im = ax.imshow(a, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")

        ax.set_title(city, fontsize=12, pad=6)
        ax.axis("off")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(cbar_label, fontsize=9)

        if show_mask:
            # Find corresponding mask
            m = None
            for (c2, m2) in masks:
                if c2 == city:
                    m = m2
                    break
            if m is not None:
                # Overlay invalid pixels (mask==0) in red with transparency
                invalid = (m < 0.5).astype(float)
                invalid[invalid == 0] = np.nan
                ax.imshow(invalid, cmap="Reds", vmin=0, vmax=1, alpha=0.35, interpolation="nearest")

    title_suffix = ""
    if dataset_key == "uhii":
        title_suffix = f" — {UHII_BANDS[uhii_band] if UHII_BANDS else uhii_band}"
    elif dataset_key == "lst":
        title_suffix = f" — {'Day' if lst_channel==0 else 'Night'}"

    fig.suptitle(f"{dataset_labels[dataset_key]}{title_suffix} — {MONTHS[month-1][0]} {year}", fontsize=16, y=0.98)
    plt.tight_layout()
    plt.show()

# ---------- 5) Wire up widgets ----------
ui = VBox([controls_row1, controls_row2])
out = interactive_output(
    render_grid,
    {
        "dataset_key": dataset_dd,
        "year": year_dd,
        "month": month_dd,
        "uhii_band": uhii_band_dd,
        "lst_channel": lst_channel_dd,
        "show_mask": show_mask_cb,
        "cities": city_sel
    }
)
display(ui, out)

print("Loaded cities:", ", ".join(sorted(city_data.keys())))
print("Tip: Use 'Overlay mask' to highlight invalid pixels (red). UHII band and LST channel selectors adapt the")



Output()

Loaded cities: Bengaluru, Chennai, Delhi, Kolkata, Mumbai, Pune
Tip: Use 'Overlay mask' to highlight invalid pixels (red). UHII band and LST channel selectors adapt the


## Precompute Landcover Features Maps
 - Reduce compute overhead by calculating feature maps of Landcover images (18 instances per city) beforehand. Just load those images and replicate for timestep, reduces latency while training.

## **Neural Network**

In [52]:
# ---------- NEW: UHII band helpers (SUHI/CUHI mapping & coverage) ----------
import re, json

_CUHI_KEYWORDS = [
    r"\bcuhi\b", r"\bcanopy\b", r"\bsat\b", r"\bsurface air temp\b", r"\bcanopy uhi\b"
]
_SUHI_KEYWORDS = [
    r"\bsuhi\b", r"\bsurface\b", r"\blst\b", r"\blst-based\b", r"\bsurface uhi\b"
]

def _classify_band(name: str) -> str:
    n = name.lower().strip()
    for pat in _CUHI_KEYWORDS:
        if re.search(pat, n):
            return "CUHI"
    for pat in _SUHI_KEYWORDS:
        if re.search(pat, n):
            return "SUHI"
    # fallback: if "sat" token exists (common in your meta), map to CUHI, else SUHI
    if "sat" in n or "canopy" in n:
        return "CUHI"
    return "SUHI"

def build_band_info(uhii_bands_order: list[str]) -> dict:
    """
    Returns:
      {
        "bands": [{"idx": i, "name": "AMOD1", "type": "SUHI"}, ...],
        "suhi_indices": [ ... ],
        "cuhi_indices": [ ... ]
      }
    """
    rows, suhi, cuhi = [], [], []
    for i, nm in enumerate(uhii_bands_order):
        btype = _classify_band(nm)
        rows.append({"idx": i, "name": nm, "type": btype})
        (suhi if btype == "SUHI" else cuhi).append(i)
    return {"bands": rows, "suhi_indices": suhi, "cuhi_indices": cuhi}

def write_city_band_coverage(city_dir: str, uhii_mask: torch.Tensor, band_info: dict):
    """
    Writes {band_name, type, valid_pixel_count} for that city to band_coverage.json.
    uhii_mask shape: (T, B, H, W)
    """
    counts = uhii_mask.sum(dim=(0,2,3)).to(torch.int64).tolist()  # length B
    out = []
    for row in band_info["bands"]:
        i = row["idx"]
        out.append({"idx": i, "name": row["name"], "type": row["type"], "valid_pixels": int(counts[i])})
    with open(os.path.join(city_dir, "band_coverage.json"), "w") as f:
        json.dump({"coverage": out}, f, indent=2)


In [53]:
import torch
from torch.utils.data import Dataset
from typing import Dict, List, Tuple, Optional, Union


def apply_shared_geometric_augs(sample: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    """Apply the same random flip/rotation to all spatial tensors in ``sample``."""
    do_vflip = torch.rand(1).item() > 0.5
    do_hflip = torch.rand(1).item() > 0.5
    do_rot90 = torch.rand(1).item() > 0.5

    keys = ["pm", "pm_mask", "lst", "lst_mask", "uhii", "uhii_mask", "lc"]
    for k in keys:
        if k not in sample or not isinstance(sample[k], torch.Tensor):
            continue
        x = sample[k]
        if do_vflip:
            x = x.flip(-2)
        if do_hflip:
            x = x.flip(-1)
        if do_rot90:
            x = torch.rot90(x, 1, [-2, -1])
        sample[k] = x
    return sample


def _accumulate_tensor_stats(t: torch.Tensor, totals: Dict[str, float]):
    if not isinstance(t, torch.Tensor):
        return
    tensor = t.detach()
    if tensor.device.type != "cpu":
        tensor = tensor.to("cpu")
    flat = tensor.reshape(-1).double()
    totals["sum"] += flat.sum().item()
    totals["sum_sq"] += flat.pow(2).sum().item()
    totals["count"] += flat.numel()


def compute_lst_normalization_stats(data: Dict[str, dict], cities: Optional[List[str]] = None) -> Tuple[float, float]:
    """Compute mean/std for LST tensors using CPU memory to avoid GPU overhead."""
    totals = {"sum": 0.0, "sum_sq": 0.0, "count": 0}
    for city in (cities or list(data.keys())):
        pack = data.get(city, {})
        lst = pack.get("lst")
        if isinstance(lst, torch.Tensor):
            _accumulate_tensor_stats(lst, totals)
    if totals["count"] == 0:
        raise RuntimeError("No LST tensors found for normalization.")
    mean = totals["sum"] / totals["count"]
    var = max(totals["sum_sq"] / totals["count"] - mean * mean, 0.0)
    std = max(var ** 0.5, 1e-6)
    return float(mean), float(std)


def _sanitize_tensor_and_mask(
    tensor: torch.Tensor,
    mask: Optional[torch.Tensor],
    *,
    fill_value: float
) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
    """Replace NaN/Inf values and drop them from the accompanying mask."""
    if not isinstance(tensor, torch.Tensor):
        return tensor, mask
    invalid = torch.isnan(tensor) | torch.isinf(tensor)
    tensor = torch.nan_to_num(tensor, nan=fill_value, posinf=fill_value, neginf=fill_value)
    if mask is not None and isinstance(mask, torch.Tensor):
        mask = torch.nan_to_num(mask, nan=0.0, posinf=0.0, neginf=0.0)
        if invalid.any():
            mask = mask * (~invalid).to(mask.dtype)
    return tensor, mask


def move_city_packs_to_device(city_data: Dict[str, dict], device: Union[torch.device, str], *,
                              non_blocking: bool = True, inplace: bool = False) -> Dict[str, dict]:
    """Move every tensor in ``city_data`` to ``device``. Returns the moved mapping."""
    target = torch.device(device)
    if inplace:
        dest = city_data
    else:
        dest = {}
    for city, pack in city_data.items():
        target_pack = pack if inplace else {}
        for key, value in pack.items():
            if isinstance(value, torch.Tensor):
                if value.device == target:
                    moved = value
                else:
                    moved = value.to(target, non_blocking=non_blocking)
                target_pack[key] = moved
            else:
                target_pack[key] = value
        if not inplace:
            dest[city] = target_pack
    return dest


class UHIMemoryEfficientDataset(Dataset):
    def __init__(self,
                 cities: List[str],
                 data: Dict[str, dict],
                 train: bool = True,
                 tw: int = 3,
                 use_precomputed_lc: bool = True,
                 lst_stats: Optional[Tuple[float, float]] = None,
                 target_device: Optional[Union[torch.device, str]] = None,
                 non_blocking: bool = True):
        self.cities = list(cities)
        self.data = data
        self.train = train
        self.tw = tw
        self.use_precomputed_lc = use_precomputed_lc
        self.non_blocking = non_blocking

        if target_device is not None:
            move_city_packs_to_device(self.data, target_device, non_blocking=non_blocking, inplace=True)

        self.device = self._infer_data_device()

        if lst_stats is not None:
            mean, std = lst_stats
        else:
            mean, std = compute_lst_normalization_stats(self.data, self.cities)
        self.lst_mean = float(mean)
        self.lst_std = max(float(std), 1e-6)

        self.lc_precomp_avail = {c: ("lc_encoded" in self.data[c]) for c in self.cities}

        self.indices = []
        for city in self.cities:
            uhii = self.data[city]["uhii"]
            self.indices.extend([(city, t) for t in range(uhii.shape[0])])

    def _infer_data_device(self) -> torch.device:
        for city in self.cities:
            pack = self.data[city]
            for key in ("pm25", "uhii", "lst"):
                tensor = pack.get(key)
                if isinstance(tensor, torch.Tensor):
                    return tensor.device
        return torch.device("cpu")

    def __len__(self) -> int:
        return len(self.indices)

    def _window_stack(self, x: torch.Tensor, t: int, per_step_ch: int) -> torch.Tensor:
        T, C, H, W = x.shape
        start = max(0, t - self.tw + 1)
        pad_left = self.tw - (t - start + 1)
        chunks = [x[start:t+1]]
        if pad_left > 0:
            chunks.insert(0, x[0:1].expand(pad_left, C, H, W))
        win = torch.cat(chunks, dim=0)
        return win.permute(1, 0, 2, 3).reshape(C * self.tw, H, W)

    def __getitem__(self, idx: int) -> Dict[str, Union[torch.Tensor, str]]:
        city, t = self.indices[idx]
        pack = self.data[city]

        pm = self._window_stack(pack['pm25'], t, 1).float()
        pm_mask = self._window_stack(pack['pm25_mask'], t, 1).float()
        pm, pm_mask = _sanitize_tensor_and_mask(pm, pm_mask, fill_value=0.0)
        pm = torch.clamp(pm, 0, 150.0) / 150.0
        pm = torch.nan_to_num(pm, nan=0.0, posinf=1.0, neginf=0.0)

        lst = self._window_stack(pack['lst'], t, 2).float()
        lst_mask = self._window_stack(pack['lst_mask'], t, 2).float()
        lst, lst_mask = _sanitize_tensor_and_mask(lst, lst_mask, fill_value=self.lst_mean)
        lst = (lst - self.lst_mean) / self.lst_std
        lst = torch.nan_to_num(lst, nan=0.0, posinf=0.0, neginf=0.0)

        if self.use_precomputed_lc and self.lc_precomp_avail.get(city, False) and ('lc_encoded' in pack):
            lc = pack['lc_encoded'][t]
        else:
            lc_src = pack.get('lc')
            if lc_src is None:
                raise KeyError(f"City {city} is missing 'lc' data")
            lc_t = lc_src[t]
            if isinstance(lc_t, torch.Tensor) and lc_t.shape[0] == 1 and lc_t.shape[-1] >= 1000:
                lc = lc_t.long()
            else:
                lc = lc_t.float() if isinstance(lc_t, torch.Tensor) else lc_t
        if isinstance(lc, torch.Tensor) and lc.dtype.is_floating_point:
            lc = torch.nan_to_num(lc, nan=0.0, posinf=0.0, neginf=0.0)

        uhii = pack['uhii'][t].float()
        uhii_mask = pack['uhii_mask'][t].float()
        uhii, uhii_mask = _sanitize_tensor_and_mask(uhii, uhii_mask, fill_value=0.0)

        sample = {
            "pm": pm.float(),
            "pm_mask": pm_mask.float(),
            "lst": lst.float(),
            "lst_mask": lst_mask.float(),
            "lc": lc,
            "uhii": uhii.float(),
            "uhii_mask": uhii_mask.float(),
            "city": city,
            "t": t,
        }

        if self.train:
            sample = apply_shared_geometric_augs(sample)

        return sample


In [54]:
import torch
import torch.nn as nn

def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.GroupNorm):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Embedding):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, use_silu=False):
        super(CBAM, self).__init__()
        act = nn.SiLU() if use_silu else nn.ReLU()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            act,
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid = nn.Sigmoid()
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, stride=1, padding=3)

        # Init MLPs with Xavier
        nn.init.xavier_uniform_(self.mlp[0].weight)
        nn.init.xavier_uniform_(self.mlp[2].weight)

    def forward(self, x):
        # Channel attention
        avg_out = self.mlp(self.avg_pool(x).squeeze(-1).squeeze(-1))
        max_out = self.mlp(self.max_pool(x).squeeze(-1).squeeze(-1))
        channel_att = self.sigmoid(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x = x * channel_att

        # Spatial attention
        avg_sp = torch.mean(x, dim=1, keepdim=True)
        max_sp, _ = torch.max(x, dim=1, keepdim=True)
        spatial_in = torch.cat([avg_sp, max_sp], dim=1)
        spatial_att = self.sigmoid(self.spatial_conv(spatial_in))
        x = x * spatial_att

        return x

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.gn1 = nn.GroupNorm(8, out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.gn2 = nn.GroupNorm(8, out_channels)
        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0) if in_channels != out_channels else None

    def forward(self, x):
        residual = x if self.shortcut is None else self.shortcut(x)
        x = self.conv1(x)
        x = self.gn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.gn2(x)
        return x + residual

class ResidualCBAMBlock(nn.Module):
    def __init__(self, channels, use_silu=False):
        super(ResidualCBAMBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)
        self.gn1 = nn.GroupNorm(8, channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)
        self.gn2 = nn.GroupNorm(8, channels)
        self.cbam = CBAM(channels, use_silu=use_silu)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.gn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.gn2(x)
        x = self.cbam(x)
        return x + residual

class LCEncoder(nn.Module):
    def __init__(self, num_classes=36, emb_dim=8):
        super(LCEncoder, self).__init__()
        self.embedding = nn.Embedding(num_classes, emb_dim)
        self.patchify = nn.Conv2d(emb_dim, 64, kernel_size=33, stride=33, padding=0)
        self.block1 = ResidualCBAMBlock(64)
        self.block2 = ResidualCBAMBlock(64)
        self.apply(init_weights)

    def forward(self, lc):
        # lc: (B, 1, 1333, 1333) int, squeeze to (B, 1333, 1333)
        lc = lc.squeeze(1)
        # Center crop to 1320x1320
        _, h, w = lc.shape
        crop_size = 1320
        start_h = (h - crop_size) // 2
        start_w = (w - crop_size) // 2
        lc_cropped = lc[:, start_h:start_h + crop_size, start_w:start_w + crop_size]
        # Embed
        B, H, W = lc_cropped.shape
        lc_flat = lc_cropped.reshape(B, -1).long()
        emb_flat = self.embedding(lc_flat)  # (B, H*W, emb_dim)
        emb = emb_flat.permute(0, 2, 1).reshape(B, -1, H, W)  # (B, emb_dim, H, W)
        # Patchify
        x = self.patchify(emb)  # (B, 64, 40, 40)
        x = self.block1(x)
        x = self.block2(x)
        return x

In [55]:
import torch.nn.functional as F

class LCFracEncoder(nn.Module):
    """
    Land-cover fractions encoder:
    - One-hot LC classes at 30m
    - Center-crop to 1320x1320
    - Average-pool 33x33 -> 40x40 to get per-class fractions
    - Project to 64ch then two CBAM residual blocks (same style as LCEncoder)
    """
    def __init__(self, num_classes: int = 36, out_ch: int = 64, precomputed: bool = False):
        super().__init__()
        self.precomputed = precomputed
        if not precomputed:
            self.num_classes = num_classes
            self.proj = nn.Conv2d(num_classes, out_ch, kernel_size=1, bias=False)
            self.gn = nn.GroupNorm(8, out_ch)
            self.act = nn.ReLU(inplace=True)
            self.block1 = ResidualCBAMBlock(out_ch)
            self.block2 = ResidualCBAMBlock(out_ch)
            self.apply(init_weights)
        else:
            print("LCFracEncoder: Precomputed mode enabled—acting as pass-through.")

    def forward(self, lc):
        if self.precomputed:
            return lc  # Already (B,64,40,40) from precompute
        # Original forward if not precomputed
        x = lc.squeeze(1).long()                 # (B,1333,1333)
        B, H, W = x.shape
        crop = 1320
        sh = (H - crop)//2
        sw = (W - crop)//2
        x = x[:, sh:sh+crop, sw:sw+crop]         # (B,1320,1320)

        # one-hot -> (B,1320,1320,num_classes) -> (B,num_classes,1320,1320)
        oh = F.one_hot(x, num_classes=self.num_classes).permute(0,3,1,2).float()

        # average-pool 33x33 stride 33 -> (B,num_classes,40,40)
        frac = F.avg_pool2d(oh, kernel_size=33, stride=33)  # class fractions per 40x40 cell

        # project & refine
        z = self.proj(frac)
        z = self.gn(z)
        z = self.act(z)
        z = self.block1(z)
        z = self.block2(z)
        return z

In [56]:
class PMEncoder(nn.Module):
    def __init__(self, tw=3):
        super(PMEncoder, self).__init__()
        self.entry = nn.Sequential(
            nn.Conv2d(tw, 16, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(8, 16),
            nn.ReLU()
        )
        self.res_block = ResidualBlock(16, 32)
        self.apply(init_weights)

    def forward(self, pm, pm_mask):
        # pm: (B, tw, 40, 40)
        x = pm
        x = self.entry(x)
        x = self.res_block(x)
        # Mask gating
        avg_mask = pm_mask.mean(dim=1).detach()  # (B, 40, 40)
        x = x * avg_mask.unsqueeze(1)  # (B, 32, 40, 40) * (B, 1, 40, 40)
        return x

In [57]:
class LSTEncoder(nn.Module):
    def __init__(self, tw=3):
        super(LSTEncoder, self).__init__()
        self.entry = nn.Sequential(
            nn.Conv2d(2 * tw, 32, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU()
        )
        self.res_block = ResidualBlock(32, 64)
        self.apply(init_weights)

    def forward(self, lst, lst_mask):
        # lst: (B, 2*tw, 40, 40)
        x = lst
        x = self.entry(x)
        x = self.res_block(x)
        # Mask gating
        avg_mask = lst_mask.mean(dim=1).detach()  # (B, 40, 40)
        x = x * avg_mask.unsqueeze(1)  # (B, 64, 40, 40) * (B, 1, 40, 40)
        return x

In [58]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple, Dict

# ------------------------------------------------------------
# Small building blocks
# ------------------------------------------------------------

def conv_gn_relu(in_ch, out_ch, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
        nn.GroupNorm(8, out_ch),
        nn.ReLU(inplace=True),
    )

import torch
import torch.nn as nn
from typing import Tuple

class DoubleConv(nn.Module):
    """
    Double convolution block for UNet: Conv2d -> BatchNorm -> GroupNorm -> ReLU -> Conv2d -> BatchNorm -> GroupNorm -> ReLU -> Dropout.

    Args:
        in_ch (int): Number of input channels.
        out_ch (int): Number of output channels.
        dropout_p (float): Dropout probability (default 0.2).
        dilations (Tuple[int, int]): Dilations for the two conv layers (default (1, 1)).
    """
    def __init__(self, in_ch: int, out_ch: int, dropout_p: float = 0.2, dilations: Tuple[int, int] = (1, 1)):
        super().__init__()
        d1, d2 = dilations

        # First conv block: Conv2d -> BatchNorm -> GroupNorm -> ReLU
        self.c1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=d1, dilation=d1, bias=False),
            nn.BatchNorm2d(out_ch),  # NEW: Add BatchNorm
            nn.GroupNorm(num_groups=8, num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

        # Second conv block: Conv2d -> BatchNorm -> GroupNorm -> ReLU
        self.c2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=d2, dilation=d2, bias=False),
            nn.BatchNorm2d(out_ch),  # NEW: Add BatchNorm
            nn.GroupNorm(num_groups=8, num_channels=out_ch),
            nn.ReLU(inplace=True),
        )

        # NEW: Dropout with increased probability
        self.dropout = nn.Dropout2d(p=dropout_p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.c1(x)
        x = self.c2(x)
        x = self.dropout(x)
        return x

# --- REPLACE your AttentionGate with this ---
class AttentionGate(nn.Module):
    """
    Size-agnostic gate: projects skip (x) and gate (g) with 1x1 convs,
    upsamples g to x's spatial size if needed, then computes an attention mask.
    """
    def __init__(self, x_ch: int, g_ch: int, inter_ch: Optional[int] = None):
        super().__init__()
        if inter_ch is None:
            inter_ch = max(1, x_ch // 2)
        self.theta_x = nn.Conv2d(x_ch, inter_ch, kernel_size=1, stride=1, bias=False)
        self.phi_g   = nn.Conv2d(g_ch, inter_ch, kernel_size=1, stride=1, bias=True)
        self.psi     = nn.Conv2d(inter_ch, 1, kernel_size=1, stride=1, bias=True)
        self.act     = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, g):
        # ensure g matches x spatially
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
        attn = self.act(self.theta_x(x) + self.phi_g(g))   # (B, inter, H, W)
        attn = self.sigmoid(self.psi(attn))                # (B, 1, H, W)
        return x * attn


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_transpose: bool = True):
        super().__init__()
        if use_transpose:
            self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        else:
            self.up = nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
            )

    def forward(self, x):
        return self.up(x)

# ------------------------------------------------------------
# Stream attention (tiny MLP producing α_lc, α_lst, α_pm)
# ------------------------------------------------------------

class StreamAttention(nn.Module):
    def __init__(self, ch_lc: int, ch_lst: int, ch_pm: int, d: int = 32):
        super().__init__()
        self.proj_lc  = nn.Conv2d(ch_lc,  d, 1, bias=False)
        self.proj_lst = nn.Conv2d(ch_lst, d, 1, bias=False)
        self.proj_pm  = nn.Conv2d(ch_pm,  d, 1, bias=False)
        self.mlp = nn.Sequential(
            nn.Linear(3*d, d*2),
            nn.ReLU(inplace=True),
            nn.Linear(2*d, 3)
        )

    def forward(self, F_lc, F_lst, F_pm):
        # GAP → project → concat → softmax
        v_lc  = self.proj_lc(F_lc ).mean(dim=(2,3))  # (B,d)
        v_lst = self.proj_lst(F_lst).mean(dim=(2,3)) # (B,d)
        v_pm  = self.proj_pm(F_pm ).mean(dim=(2,3))  # (B,d)
        v = torch.cat([v_lc, v_lst, v_pm], dim=1)    # (B,3d)
        alpha = torch.softmax(self.mlp(v), dim=1)    # (B,3)
        a_lc, a_lst, a_pm = alpha[:,0:1], alpha[:,1:2], alpha[:,2:3]
        F_lc  = F_lc  * a_lc.unsqueeze(-1).unsqueeze(-1)
        F_lst = F_lst * a_lst.unsqueeze(-1).unsqueeze(-1)
        F_pm  = F_pm  * a_pm.unsqueeze(-1).unsqueeze(-1)
        return F_lc, F_lst, F_pm, alpha

# ------------------------------------------------------------
# U-Net (plain or attention) with configurable depth and dilations
# Input: fused channels @ 40×40
# ------------------------------------------------------------

## Corrected UNetBackbone Class (Definitive Version)
class UNetBackbone(nn.Module):
    def __init__(
        self,
        in_ch: int,
        base: int = 128,
        depth: int = 4,
        use_attn_gates: bool = True,
        bottleneck_dilations: Tuple[int,int] = (1,1),
        dropout_p: float = 0.1,
        use_transpose_up: bool = True,
        out_ch_after_decoder: int = 256,
    ):
        super().__init__()
        assert depth in (3,4), "depth must be 3 or 4 for a 40×40 grid"
        self.depth = depth
        self.use_attn = use_attn_gates
        self.out_ch = out_ch_after_decoder

        # Encoder widths
        c0 = base      # 128 @ 40x40
        c1 = base * 2  # 256 @ 20x20
        c2 = base * 3  # 384 @ 10x10
        c3 = base * 4  # 512 @ 5x5 (depth 4 only)
        cb = base * 6  # 768 (bottleneck)

        # --- Encoder ---
        self.enc0 = DoubleConv(in_ch, c0)
        self.down1 = nn.MaxPool2d(2)
        self.enc1 = DoubleConv(c0, c1)
        self.down2 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(c1, c2)
        if depth == 4:
            self.down3 = nn.MaxPool2d(2)
            self.enc3 = DoubleConv(c2, c3)
            self.bot = DoubleConv(c3, cb, dropout_p=dropout_p, dilations=bottleneck_dilations)
        else: # depth == 3
            self.bot = DoubleConv(c2, cb, dropout_p=dropout_p, dilations=bottleneck_dilations)

        # --- Decoder (Corrected Channel Logic) ---
        if depth == 4:
            self.up3 = UpBlock(cb, c3, use_transpose=use_transpose_up)
            self.dec3 = DoubleConv(c3 + c2, c2) # Concatenates upsampled 'b' and skip 'e2'

        up2_in_ch = c2 if depth == 4 else cb
        self.up2 = UpBlock(up2_in_ch, c1, use_transpose=use_transpose_up)
        self.dec2 = DoubleConv(c1 + c1, c1) # Concatenates upsampled 'd3' or 'b' and skip 'e1'

        self.up1 = UpBlock(c1, c0, use_transpose=use_transpose_up)
        self.dec1 = DoubleConv(c0 + c0, c0) # Concatenates upsampled 'd2' and skip 'e0'

        self.dec0 = DoubleConv(c0 + c0, self.out_ch) # Concatenates 'd1' and skip 'e0'

        # --- Attention Gates ---
        if self.use_attn:
            if depth == 4:
                # The gate for skip e2 is gated by the upsampled bottleneck
                self.g2 = AttentionGate(x_ch=c2, g_ch=c3)
            else: # depth=3
                # The gate for skip e1 is gated by the bottleneck
                self.g1_d3 = AttentionGate(x_ch=c1, g_ch=cb)
            # Common gates
            self.g1_d4 = AttentionGate(x_ch=c1, g_ch=c2)
            self.g0 = AttentionGate(x_ch=c0, g_ch=c1)

    def forward(self, x):
        # --- Encoder Path ---
        e0 = self.enc0(x)                # 40x40, c0
        e1 = self.enc1(self.down1(e0))   # 20x20, c1
        e2 = self.enc2(self.down2(e1))   # 10x10, c2

        # --- Bottleneck ---
        if self.depth == 4:
            e3 = self.enc3(self.down3(e2)) # 5x5, c3
            b = self.bot(e3)              # 5x5, cb
        else: # depth == 3
            b = self.bot(e2)              # 10x10, cb

        # --- Decoder Path (Corrected Skip Connection Logic) ---
        if self.depth == 4:
            # Upsample from 5x5 -> 10x10. Concat with e2 (10x10 skip).
            d_up3 = self.up3(b)
            s2 = self.g2(x=e2, g=d_up3) if self.use_attn else e2
            d3 = self.dec3(torch.cat([d_up3, s2], dim=1)) # -> c2 channels @ 10x10

            # Upsample from 10x10 -> 20x20. Concat with e1 (20x20 skip).
            d_up2 = self.up2(d3)
            s1 = self.g1_d4(x=e1, g=d3) if self.use_attn else e1
            d2 = self.dec2(torch.cat([d_up2, s1], dim=1)) # -> c1 channels @ 20x20
        else: # depth == 3
            # Upsample from 10x10 -> 20x20. Concat with e1 (20x20 skip).
            d_up2 = self.up2(b)
            s1 = self.g1_d3(x=e1, g=b) if self.use_attn else e1
            d2 = self.dec2(torch.cat([d_up2, s1], dim=1)) # -> c1 channels @ 20x20

        # Upsample from 20x20 -> 40x40. Concat with e0 (40x40 skip).
        d_up1 = self.up1(d2)
        s0 = self.g0(x=e0, g=d2) if self.use_attn else e0
        d1 = self.dec1(torch.cat([d_up1, s0], dim=1)) # -> c0 channels @ 40x40

        # Final conv block at full resolution (no upsampling).
        # Note: The original U-Net has one less skip connection. This is a common variant.
        # To match the __init__, we need one more concat. We reuse s0.
        d0 = self.dec0(torch.cat([d1, s0], dim=1))

        return d0
# ------------------------------------------------------------
# Config and Top-level model
# ------------------------------------------------------------

@dataclass
class UHINetConfig:
    # stream usage
    use_lc: bool = True
    use_lst: bool = True
    use_pm: bool  = True

    # encoders (channels)
    ch_lc: int = 64
    ch_lst: int = 64
    ch_pm:  int = 32

    # stream attention
    use_stream_attn: bool = True

    # fusion + unet
    depth: int = 4                 # 3 or 4
    use_unet_attn: bool = True
    base_unet_width: int = 128
    out_feat_width: int = 256
    bottleneck_dilations: Tuple[int,int] = (1,1)
    dropout_p: float = 0.1
    use_transpose_up: bool = True

    # head
    out_bands: int = 8             # set from meta at runtime if needed

class UHINet(nn.Module):
    def __init__(
        self,
        cfg: UHINetConfig,
        lc_encoder: Optional[nn.Module] = None,
        lst_encoder: Optional[nn.Module] = None,
        pm_encoder: Optional[nn.Module]  = None,
    ):
        super().__init__()
        self.cfg = cfg

        # Encoders (use provided ones or expect external set via attributes)
        self.lc_encoder  = lc_encoder
        self.lst_encoder = lst_encoder
        self.pm_encoder  = pm_encoder

        # Fused channels
        fused_in = 0
        if cfg.use_lc:  fused_in += cfg.ch_lc
        if cfg.use_lst: fused_in += cfg.ch_lst
        if cfg.use_pm:  fused_in += cfg.ch_pm

        # Optional stream attention
        if cfg.use_stream_attn and (cfg.use_lc and cfg.use_lst and cfg.use_pm):
            self.stream_attn = StreamAttention(cfg.ch_lc, cfg.ch_lst, cfg.ch_pm, d=32)
        else:
            self.stream_attn = None

        # U-Net backbone
        self.unet = UNetBackbone(
            in_ch=fused_in,
            base=cfg.base_unet_width,
            depth=cfg.depth,
            use_attn_gates=cfg.use_unet_attn,
            bottleneck_dilations=cfg.bottleneck_dilations,
            dropout_p=cfg.dropout_p,
            use_transpose_up=cfg.use_transpose_up,
            out_ch_after_decoder=cfg.out_feat_width,
        )

        # 1×1 head to bands
        self.head = nn.Conv2d(cfg.out_feat_width, cfg.out_bands, kernel_size=1, bias=True)
        nn.init.kaiming_normal_(self.head.weight, mode='fan_out', nonlinearity='relu')
        nn.init.constant_(self.head.bias, 0.0)

    def forward(
        self,
        pm:  Optional[torch.Tensor] = None,      # (B, tw, 40, 40)
        pm_mask: Optional[torch.Tensor] = None,  # (B, tw, 40, 40)
        lst: Optional[torch.Tensor] = None,      # (B, 2*tw, 40, 40)
        lst_mask: Optional[torch.Tensor] = None, # (B, 2*tw, 40, 40)
        lc:  Optional[torch.Tensor] = None,      # (B, 1, 1333, 1333) int16
        return_alpha: bool = False,
    ) -> Dict[str, torch.Tensor]:

        feats = []
        alpha = None

        if self.cfg.use_lc:
            assert self.lc_encoder is not None, "LC encoder not set"
            F_lc = self.lc_encoder(lc)                     # (B, ch_lc, 40, 40)
        else:
            F_lc = None

        if self.cfg.use_lst:
            assert self.lst_encoder is not None, "LST encoder not set"
            F_lst = self.lst_encoder(lst, lst_mask)        # (B, ch_lst, 40, 40)
        else:
            F_lst = None

        if self.cfg.use_pm:
            assert self.pm_encoder is not None, "PM encoder not set"
            F_pm = self.pm_encoder(pm, pm_mask)            # (B, ch_pm, 40, 40)
        else:
            F_pm = None

        if self.stream_attn is not None:
            F_lc, F_lst, F_pm, alpha = self.stream_attn(F_lc, F_lst, F_pm)
            feats = [F_lc, F_lst, F_pm]
        else:
            if F_lc is not None:  feats.append(F_lc)
            if F_lst is not None: feats.append(F_lst)
            if F_pm is not None:  feats.append(F_pm)

        F0 = torch.cat(feats, dim=1)                      # (B, fused_in, 40, 40)
        Fd = self.unet(F0)                                # (B, out_feat_width, 40, 40)
        yhat = self.head(Fd)                              # (B, bands, 40, 40)

        if return_alpha and alpha is not None:
            return {"yhat": yhat, "alpha": alpha, "features": Fd}
        return {"yhat": yhat, "features": Fd}


In [59]:
# assemble_uhi_system.py

import os, json, math
from dataclasses import dataclass, asdict
from typing import Optional, Tuple, List, Dict

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.cuda.amp import GradScaler, autocast


# ---- bring your classes into scope (edit the import paths as per your project layout)
# from encoders import LCEncoder, PMEncoder, LSTEncoder
# from model_uhi import UHINet, UHINetConfig
# If you defined them in the same notebook/file, you can skip the imports.

# ----------------------------
# Utilities
# ----------------------------
def read_bands_from_meta(root_dir: str, cities_priority: Optional[List[str]] = None) -> int:
    """
    Reads the first available city's meta.json and returns the number of UHII bands (B).
    """
    cities = [d for d in sorted(os.listdir(root_dir)) if os.path.isdir(os.path.join(root_dir, d))]
    if cities_priority:
        # keep order but ensure existence
        cities = [c for c in cities_priority if c in cities] + [c for c in cities if c not in (cities_priority or [])]
    for c in cities:
        meta_p = os.path.join(root_dir, c, "meta.json")
        if os.path.exists(meta_p):
            with open(meta_p, "r") as f:
                meta = json.load(f)
            uh_order = meta.get("uhii_bands_order", [])
            if uh_order:
                return len(uh_order)
            # fallback: read uhii tensor to infer channels
            uh_p = os.path.join(root_dir, c, meta.get("files", {}).get("uhii", "uhii.pt"))
            if os.path.exists(uh_p):
                uh = torch.load(uh_p, map_location="cpu")
                return uh.shape[1]  # (T, B, 40, 40)
    raise RuntimeError(f"Could not infer UHII band count from {root_dir} (no meta/uhii found).")


def seed_everything(seed: int = 42, deterministic: bool = False):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = True


def exclude_from_weight_decay(n: str, p: nn.Parameter) -> bool:
    """
    Returns True for parameters that should have NO weight decay:
    - biases
    - norm scales (BatchNorm/GroupNorm/LayerNorm/InstanceNorm)
    - embedding weights (optional but usually helpful)
    """
    if p.ndim == 1:  # common for norm/bias vectors
        return True
    if n.endswith(".bias"):
        return True
    norm_keywords = ["norm", "gn", "bn", "ln", "in", "running_mean", "running_var"]
    if any(k in n.lower() for k in norm_keywords):
        return True
    if "embedding" in n.lower():
        return True
    return False


def make_optimizer(model: nn.Module, lr: float = 3e-4, weight_decay: float = 1e-2) -> AdamW:
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if exclude_from_weight_decay(n, p) else decay).append(p)
    param_groups = [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return AdamW(param_groups, lr=lr, betas=(0.9, 0.999))


def make_warmup_cosine_scheduler(optimizer: torch.optim.Optimizer,
                                 total_steps: int,
                                 warmup_steps: int = 0):
    """
    Per-step LR scheduler with (optional) linear warmup and cosine decay to 0.
    Use it by calling `scheduler.step()` each optimizer step.
    """
    warmup_steps = max(0, warmup_steps)
    total_steps = max(1, total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return LambdaLR(optimizer, lr_lambda)


def count_params(model: nn.Module) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": train}


# ----------------------------
# High-level Config (wraps UHINetConfig and encoder knobs)
# ----------------------------
from dataclasses import dataclass, asdict
from typing import Optional, Tuple, List, Dict
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.cuda.amp import GradScaler

@dataclass
class SystemConfig:
    lr: float = 3e-5
    weight_decay: float = 1e-3
    grad_clip_norm: float = 0.5
    warmup_steps: int = 0
    epochs: int = 50
    batch_size: int = 32
    optimizer: str = "adamw"
    scheduler: str = "cosine"
    use_lc: bool = True
    use_lst: bool = True
    use_pm: bool = True
    use_stream_attn: bool = True
    use_unet_attn: bool = True
    depth: int = 4
    base_unet_width: int = 128
    out_feat_width: int = 256
    bottleneck_dilations: Tuple[int, int] = (1, 1)
    use_transpose_up: bool = True
    out_bands: int = 0
    dropout_p: float = 0.1
    tw: int = 3
    lc_encoder_type: str = "cbam"
    ch_lc: int = 64
    ch_lst: int = 64
    ch_pm: int = 32
    num_lc_classes: int = 36
    lc_emb_dim: int = 8
    num_workers: int = 0
    pin_memory: bool = True
    seed: int = 42
    amp: bool = True
    total_steps: Optional[int] = None

    def __post_init__(self):
        assert self.lr > 0
        assert 0 <= self.grad_clip_norm
        assert self.depth in (3, 4)
        assert 0.0 <= self.dropout_p <= 1.0


# ----------------------------
# Builder
# ----------------------------
def build_encoders(cfg: SystemConfig, lc_precomputed: bool | None = None):
    """
    PATCHED: optional lc_precomputed flag controls LCFracEncoder mode.
    If None, default to 'False' (raw 30 m to fractions).
    """
    if cfg.lc_encoder_type == "cbam":
        lc_enc = LCEncoder(num_classes=cfg.num_lc_classes, emb_dim=cfg.lc_emb_dim)
    elif cfg.lc_encoder_type == "frac":
        use_pre = bool(lc_precomputed)
        lc_enc = LCFracEncoder(num_classes=cfg.num_lc_classes, out_ch=cfg.ch_lc, precomputed=use_pre)
    else:
        raise ValueError(f"Unknown lc_encoder_type: {cfg.lc_encoder_type}")

    pm_enc  = PMEncoder(tw=cfg.tw)
    lst_enc = LSTEncoder(tw=cfg.tw)
    return lc_enc, pm_enc, lst_enc



def build_uhi_model(cfg: SystemConfig,
                    out_bands: int,
                    lc_enc: Optional[nn.Module],
                    lst_enc: Optional[nn.Module],
                    pm_enc: Optional[nn.Module]) -> nn.Module:
    # Prepare UHINetConfig
    core_cfg = UHINetConfig(
        use_lc=cfg.use_lc,
        use_lst=cfg.use_lst,
        use_pm=cfg.use_pm,
        ch_lc=cfg.ch_lc,
        ch_lst=cfg.ch_lst,
        ch_pm=cfg.ch_pm,
        use_stream_attn=cfg.use_stream_attn,
        depth=cfg.depth,
        use_unet_attn=cfg.use_unet_attn,
        base_unet_width=cfg.base_unet_width,
        out_feat_width=cfg.out_feat_width,
        bottleneck_dilations=cfg.bottleneck_dilations,
        dropout_p=cfg.dropout_p,
        use_transpose_up=cfg.use_transpose_up,
        out_bands=out_bands
    )
    model = UHINet(core_cfg, lc_encoder=lc_enc if cfg.use_lc else None,
                   lst_encoder=lst_enc if cfg.use_lst else None,
                   pm_encoder=pm_enc if cfg.use_pm else None)
    return model

def detect_lc_precompute_availability(root_dir: str, cities: List[str]) -> tuple[Dict[str, bool], bool]:
    """
    Returns: (per_city_available, all_available)
    """
    avail = {}
    for c in cities:
        p = os.path.join(root_dir, c, "lc_encoded.pt")
        avail[c] = os.path.exists(p)
    all_ok = all(avail.values())
    return avail, all_ok


def build_system(root_dir: str,
                 cfg: SystemConfig,
                 cities_priority: Optional[List[str]] = None,
                 device: Optional[torch.device] = None):
    # infer bands
    out_bands = cfg.out_bands or read_bands_from_meta(root_dir, cities_priority)

    # determine city list (from folders)
    cities = [d for d in sorted(os.listdir(root_dir))
              if os.path.isdir(os.path.join(root_dir, d))]
    per_city_avail, all_avail = detect_lc_precompute_availability(root_dir, cities)

    # encoders (tie precomputed mode to actual files)
    lc_enc, pm_enc, lst_enc = build_encoders(cfg, lc_precomputed=all_avail)

    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_uhi_model(cfg, out_bands, lc_enc, lst_enc, pm_enc).to(device)

    optimizer = make_optimizer(model, lr=cfg.lr, weight_decay=cfg.weight_decay)
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler(
            device="cuda" if torch.cuda.is_available() else "cpu",
            enabled=cfg.amp,
        )
    else:
        scaler = GradScaler(enabled=cfg.amp)

    sys = {
        "model": model,
        "optimizer": optimizer,
        "scheduler": None,
        "scaler": scaler,
        "cfg": asdict(cfg),
        "param_counts": count_params(model),
        "device": device,
        "lc_precompute_available": per_city_avail,  # <-- keep for reference
    }
    return sys


In [60]:
def precompute_lc_features(
    data: Dict[str, dict],  # Your city_data dict
    cfg: SystemConfig,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    overwrite: bool = False
):
    """
    Precomputes LC encoded features (e.g., 64x40x40 per annual) for all cities, replicates to monthly (216), and saves as 'lc_encoded.pt' per city.
    """
    # Build LC encoder based on cfg (no PM/LST needed)
    if cfg.lc_encoder_type == "cbam":
        lc_enc = LCEncoder(num_classes=cfg.num_lc_classes, emb_dim=cfg.lc_emb_dim)
    elif cfg.lc_encoder_type == "frac":
        lc_enc = LCFracEncoder(num_classes=cfg.num_lc_classes, out_ch=cfg.ch_lc)
    else:
        raise ValueError(f"Unsupported lc_encoder_type for precompute: {cfg.lc_encoder_type}")

    lc_enc = lc_enc.to(device)
    lc_enc.eval()

    for city, pack in data.items():
        city_dir = os.path.join(ROOT_DIR, city)  # Assuming ROOT_DIR is your tensor root
        encoded_path = os.path.join(city_dir, "lc_encoded.pt")
        if os.path.exists(encoded_path) and not overwrite:
            print(f"Skipping {city}: lc_encoded.pt exists.")
            continue

        lc_annual = pack['lc_annual']  # (18,1,1333,1333) int
        Y = lc_annual.shape[0]
        encoded_annual = []  # List for 18 encoded

        with torch.no_grad():
            for y in range(Y):
                lc_y = lc_annual[y:y+1].to(device)  # (1,1,1333,1333)
                feat = lc_enc(lc_y)  # (1,64,40,40)
                encoded_annual.append(feat.cpu())

        encoded_annual = torch.cat(encoded_annual, dim=0)  # (18,64,40,40)

        # Replicate to monthly (repeat_interleave 12, trim to T=216)
        encoded_monthly = encoded_annual.repeat_interleave(12, dim=0)[:T].contiguous()  # (216,64,40,40)

        torch.save(encoded_monthly, encoded_path)
        print(f"Precomputed and saved lc_encoded for {city}: {encoded_monthly.shape}")

        # Update meta.json (optional but recommended)
        meta_path = os.path.join(city_dir, "meta.json")
        if os.path.exists(meta_path):
            with open(meta_path, "r") as f:
                meta = json.load(f)
            meta["files"]["lc_encoded"] = "lc_encoded.pt"
            meta["lc_encoded_shape"] = [int(s) for s in encoded_monthly.shape]
            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=2)

In [61]:
# training_loop.py
import os, json, csv, time, math
from typing import Dict, List, Optional
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast
import matplotlib.pyplot as plt

# =========================
# Masked metrics & loss
# =========================
def masked_mae(yhat: torch.Tensor, y: torch.Tensor, m: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    num = torch.sum(torch.abs(yhat - y) * m)
    den = torch.sum(m) + eps
    return num / den

def masked_rmse(yhat: torch.Tensor, y: torch.Tensor, m: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    num = torch.sum((yhat - y) ** 2 * m)
    den = torch.sum(m) + eps
    return torch.sqrt(num / den)

def _gaussian_window(window_size=11, sigma=1.5, channels=1, device="cpu", dtype=torch.float32):
    gauss = torch.tensor([math.exp(-(x - window_size//2)**2/(2*sigma**2)) for x in range(window_size)],
                         dtype=dtype, device=device)
    gauss = (gauss / gauss.sum()).unsqueeze(0)
    window_2d = (gauss.T @ gauss).unsqueeze(0).unsqueeze(0)
    return window_2d.expand(channels, 1, window_size, window_size).contiguous()

def masked_ssim(yhat: torch.Tensor, y: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    B, C, _, _ = y.shape
    # Use float32 for SSIM calculation regardless of input dtype for stability
    yhat_f32 = yhat.to(torch.float32)
    y_f32 = y.to(torch.float32)

    window = _gaussian_window(11, 1.5, channels=C, device=y.device, dtype=torch.float32)
    C1, C2 = 0.01**2, 0.03**2

    mu_x = F.conv2d(yhat_f32, window, padding=5, groups=C)
    mu_y = F.conv2d(y_f32, window, padding=5, groups=C)
    mu_x2, mu_y2, mu_xy = mu_x*mu_x, mu_y*mu_y, mu_x*mu_y

    sigma_x2 = F.conv2d(yhat_f32*yhat_f32, window, padding=5, groups=C) - mu_x2
    sigma_y2 = F.conv2d(y_f32*y_f32, window, padding=5, groups=C) - mu_y2
    sigma_xy = F.conv2d(yhat_f32*y_f32, window, padding=5, groups=C) - mu_xy

    num = (2*mu_xy + C1) * (2*sigma_xy + C2)
    den = (mu_x2 + mu_y2 + C1) * (sigma_x2 + sigma_y2 + C2)
    ssim_vals = (num / den).mean(dim=(1,2,3))

    valid_ratio = (m.view(B, -1).mean(dim=1)).clamp(min=1e-6)
    return (ssim_vals * valid_ratio).mean()

import torch
from torch import Tensor
from typing import Dict, Tuple

def compute_loss(yhat: Tensor, y: Tensor, m: Tensor, lambda_ssim: float = 0.0) -> Tuple[Tensor, Dict[str, float]]:
    """
    Compute masked loss (MAE + optional SSIM) for UHI prediction.

    Args:
        yhat (Tensor): Predicted UHI tensor [batch, channels, height, width]
        y (Tensor): Ground truth UHI tensor [batch, channels, height, width]
        m (Tensor): Mask tensor [batch, channels, height, width]
        lambda_ssim (float): Weight for SSIM loss term (default 0.0 to disable)

    Returns:
        Tuple[Tensor, Dict[str, float]]: Loss value and metrics dict
    """
    assert yhat.shape[1] == y.shape[1] == m.shape[1], \
        f"Channel mismatch: yhat={tuple(yhat.shape)}, y={tuple(y.shape)}, m={tuple(m.shape)}"
    # Compute masked MAE
    mae = torch.abs(yhat - y) * m
    mae = mae.sum() / m.sum().clamp(min=1e-6)  # Avoid division by zero

    # Initialize metrics dict
    metrics = {"mae": mae.item()}

    if lambda_ssim > 0:
        # Compute masked SSIM (assumes you have a masked_ssim implementation)
        ssim_val = masked_ssim(yhat, y, m)  # Your existing function
        loss = mae + lambda_ssim * (1.0 - ssim_val)
        metrics["ssim"] = ssim_val.item()
        # NEW: Debug print to track loss components
        print(f"Debug: MAE={mae:.3f}, SSIM={ssim_val:.3f}, Total Loss={loss:.3f}")
        return loss, metrics

    # NEW: Default case (no SSIM)
    print(f"Debug: MAE={mae:.3f} (SSIM disabled)")
    return mae, metrics

In [62]:
# =========================
# One epoch train / eval
# =========================
from torch.cuda.amp import autocast
import torch.nn.functional as F

def train_one_epoch(
    model: nn.Module,
    dataloader,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    scaler: torch.cuda.amp.GradScaler,
    epoch: int,
    steps_per_epoch: int,
    device: torch.device,
    lambda_ssim: float = 0.0,
    grad_clip_norm: float = 0.5,
    warmup_steps: int = 0,
):
    model.train()
    tot_loss, tot_mae = 0.0, 0.0
    n_batches = 0

    def _move_tensor(x):
        return x if not isinstance(x, torch.Tensor) or x.device == device else x.to(device, non_blocking=True)

    for step, batch in enumerate(dataloader):
        pm = _move_tensor(batch["pm"])
        pm_m = _move_tensor(batch["pm_mask"])
        lst = _move_tensor(batch["lst"])
        lst_m = _move_tensor(batch["lst_mask"])
        lc = _move_tensor(batch["lc"]).long()
        y = _move_tensor(batch["uhii"])
        m = _move_tensor(batch["uhii_mask"])


        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=scaler.is_enabled()):
            out = model(pm=pm, pm_mask=pm_m, lst=lst, lst_mask=lst_m, lc=lc)
            yhat = out["yhat"]
            loss, metrics = compute_loss(yhat, y, m, lambda_ssim=lambda_ssim)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        tot_loss += loss.item()
        tot_mae  += metrics["mae"]
        n_batches += 1

    return {"loss": tot_loss / max(1, n_batches), "mae": tot_mae / max(1, n_batches)}

@torch.no_grad()
def evaluate_epoch(model, loader, device, lambda_ssim: float = 0.0):
    model.eval()
    agg = {"loss": 0.0, "mae": 0.0, "rmse": 0.0, "ssim": 0.0}
    n = 0
    for batch in loader:
        pm = batch["pm"] if not isinstance(batch["pm"], torch.Tensor) or batch["pm"].device == device else batch["pm"].to(device, non_blocking=True)
        pm_m = batch["pm_mask"] if not isinstance(batch["pm_mask"], torch.Tensor) or batch["pm_mask"].device == device else batch["pm_mask"].to(device, non_blocking=True)
        lst = batch["lst"] if not isinstance(batch["lst"], torch.Tensor) or batch["lst"].device == device else batch["lst"].to(device, non_blocking=True)
        lst_m = batch["lst_mask"] if not isinstance(batch["lst_mask"], torch.Tensor) or batch["lst_mask"].device == device else batch["lst_mask"].to(device, non_blocking=True)
        lc = batch["lc"] if not isinstance(batch["lc"], torch.Tensor) or batch["lc"].device == device else batch["lc"].to(device, non_blocking=True)
        lc = lc.long()
        y = batch["uhii"] if not isinstance(batch["uhii"], torch.Tensor) or batch["uhii"].device == device else batch["uhii"].to(device, non_blocking=True)
        m = batch["uhii_mask"] if not isinstance(batch["uhii_mask"], torch.Tensor) or batch["uhii_mask"].device == device else batch["uhii_mask"].to(device, non_blocking=True)

        out = model(pm=pm, pm_mask=pm_m, lst=lst, lst_mask=lst_m, lc=lc)
        yhat = out["yhat"]

        loss, _ = compute_loss(yhat, y, m, lambda_ssim=lambda_ssim)
        mae  = masked_mae(yhat, y, m)
        rmse = masked_rmse(yhat, y, m)
        ssim_val = masked_ssim(yhat, y, m)

        agg["loss"] += loss.item(); agg["mae"] += mae.item()
        agg["rmse"] += rmse.item(); agg["ssim"] += ssim_val.item()
        n += 1

    for k in agg: agg[k] /= max(1, n)
    return agg


# =========================
# Logging, plots, fit()
# =========================
def _ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def _save_history_csv(history: Dict[str, List[float]], path: str):
    keys = list(history.keys())
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["epoch"] + keys)
        for i in range(len(history[keys[0]])):
            w.writerow([i+1] + [history[k][i] for k in keys])

def _plot_curves(history: Dict[str, List[float]], out_png: str, title: str = "Training Curves"):
    plt.figure(figsize=(8,5))
    if "train_loss" in history: plt.plot(history["train_loss"], label="train_loss")
    if "val_loss" in history:   plt.plot(history["val_loss"],   label="val_loss")
    if "val_mae" in history:    plt.plot(history["val_mae"],    label="val_mae")
    if "val_rmse" in history:   plt.plot(history["val_rmse"],   label="val_rmse")
    if "val_ssim" in history:   plt.plot(history["val_ssim"],   label="val_ssim")
    plt.xlabel("epoch"); plt.ylabel("value"); plt.title(title); plt.legend(); plt.tight_layout()
    plt.savefig(out_png, dpi=130); plt.close()

def fit(
    system,
    train_loader,
    val_loader,
    out_dir: str,
    epochs: int = 20,
    lambda_ssim: float = 0.2,
    early_stop_patience: int = 10
):
    """
    PATCHED: scheduler is created here once we know the total number of steps.
    """
    _ensure_dir(out_dir)
    steps_per_epoch = max(1, len(train_loader))

    # (Re)create scheduler now that we know how many steps exist
    total_steps = epochs * steps_per_epoch
    if system.get("scheduler", None) is None:
        warmup_steps = system["cfg"].get("warmup_steps", 0)
        system["scheduler"] = make_warmup_cosine_scheduler(system["optimizer"], total_steps, warmup_steps)

    model, optimizer = system["model"], system["optimizer"]
    scheduler, scaler = system["scheduler"], system["scaler"]
    device = system["device"]

    if "cfg" in system:
        _ensure_dir(out_dir)
        with open(os.path.join(out_dir, "config.json"), "w") as f:
            json.dump(system["cfg"], f, indent=2)

    best_val_mae = float("inf")
    best_path = os.path.join(out_dir, "best.pt")
    last_path = os.path.join(out_dir, "last.pt")
    history = {
        "train_loss": [], "train_mae": [],
        "val_loss": [], "val_mae": [], "val_rmse": [], "val_ssim": [], "lr": []
    }
    patience = 0

    for epoch in range(1, epochs+1):
        t0 = time.time()
        train_stats = train_one_epoch(
            model, train_loader, optimizer, scheduler, scaler,
            epoch=epoch-1, steps_per_epoch=steps_per_epoch,
            device=device, lambda_ssim=lambda_ssim,
            grad_clip_norm=system["cfg"].get("grad_clip_norm", 0.5),
            warmup_steps=system["cfg"].get("warmup_steps", 0),
        )
        val_stats = evaluate_epoch(model, val_loader, device, lambda_ssim=0.0) if val_loader is not None else {
            "loss": float('nan'), "mae": float('nan'), "rmse": float('nan'), "ssim": float('nan')
        }

        history["train_loss"].append(train_stats["loss"])
        history["train_mae"].append(train_stats["mae"])
        history["val_loss"].append(val_stats["loss"])
        history["val_mae"].append(val_stats["mae"])
        history["val_rmse"].append(val_stats["rmse"])
        history["val_ssim"].append(val_stats["ssim"])
        history["lr"].append(optimizer.param_groups[0]["lr"])

        dt = time.time() - t0
        print(f"Epoch {epoch:03d} | "
              f"train: loss {train_stats['loss']:.4f}, mae {train_stats['mae']:.4f} | "
              f"val: loss {val_stats['loss']:.4f}, mae {val_stats['mae']:.4f}, rmse {val_stats['rmse']:.4f}, ssim {val_stats['ssim']:.4f} | "
              f"lr {history['lr'][-1]:.2e} | {dt:.1f}s")

        torch.save({"model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict()}, last_path)

        if val_stats["mae"] < best_val_mae - 1e-6:
            best_val_mae = val_stats["mae"]
            patience = 0
            torch.save({"model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict()}, best_path)
        else:
            patience += 1

        if patience >= early_stop_patience:
            print(f"Early stopping at epoch {epoch} (no val MAE improvement for {early_stop_patience} epochs).")
            break

        _save_history_csv(history, os.path.join(out_dir, "history.csv"))
        _plot_curves(history, os.path.join(out_dir, "curves.png"),
                     title="Loss/Metric Curves (train vs val)")

    _save_history_csv(history, os.path.join(out_dir, "history.csv"))
    _plot_curves(history, os.path.join(out_dir, "curves.png"),
                 title="Loss/Metric Curves (train vs val)")
    return history, best_val_mae


In [63]:
import os, json, math, csv, random
from dataclasses import replace
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import matplotlib.pyplot as plt
import numpy as np


In [64]:
# ---------- Helpers: basic masked metrics ----------

def _masked_mean(x: torch.Tensor, m: torch.Tensor, eps: float = 1e-6):
    return (x * m).sum().item() / (m.sum().item() + eps)

def metric_mae(yhat, y, m):
    return _masked_mean(torch.abs(yhat - y), m)

def metric_rmse(yhat, y, m):
    num = ((yhat - y) ** 2 * m).sum().item()
    den = (m.sum().item() + 1e-6)
    return math.sqrt(num / den)

def _gaussian_window(window_size=11, sigma=1.5, channels=1, device="cpu", dtype=torch.float32):
    gauss = torch.tensor([math.exp(-(x - window_size//2)**2/(2*sigma**2)) for x in range(window_size)],
                         dtype=dtype, device=device)
    gauss = (gauss / gauss.sum()).unsqueeze(0)
    window_2d = (gauss.T @ gauss).unsqueeze(0).unsqueeze(0)  # (1,1,11,11)
    return window_2d.expand(channels, 1, window_size, window_size).contiguous()

@torch.no_grad()
def metric_ssim(yhat: torch.Tensor, y: torch.Tensor, m: torch.Tensor) -> float:
    # Approx masked SSIM: compute full-map SSIM then scale by valid ratio
    B, C, H, W = y.shape
    window = _gaussian_window(11, 1.5, channels=C, device=y.device, dtype=y.dtype)
    mu_x = F.conv2d(yhat, window, padding=5, groups=C)
    mu_y = F.conv2d(y,    window, padding=5, groups=C)
    mu_x2, mu_y2, mu_xy = mu_x*mu_x, mu_y*mu_y, mu_x*mu_y
    sigma_x2 = F.conv2d(yhat*yhat, window, padding=5, groups=C) - mu_x2
    sigma_y2 = F.conv2d(y*y,       window, padding=5, groups=C) - mu_y2
    sigma_xy = F.conv2d(yhat*y,    window, padding=5, groups=C) - mu_xy
    C1, C2 = 0.01**2, 0.03**2
    ssim_map = ((2*mu_xy + C1)*(2*sigma_xy + C2))/((mu_x2 + mu_y2 + C1)*(sigma_x2 + sigma_y2 + C2))
    ssim_val = ssim_map.mean(dim=(1,2,3))  # (B,)
    valid_ratio = (m.view(B, -1).mean(dim=1)).clamp(min=1e-6)
    return float((ssim_val * valid_ratio).mean().item())

In [65]:
# ---------- Pearson r & R2 with masking (streaming accumulators) ----------

class CorrAccumulator:
    def __init__(self, bands: int, device: torch.device):
        self.device = device
        self.bands = bands
        self.n = torch.zeros(bands, device=device, dtype=torch.float64)
        self.sumx = torch.zeros(bands, device=device, dtype=torch.float64)
        self.sumy = torch.zeros(bands, device=device, dtype=torch.float64)
        self.sumx2 = torch.zeros(bands, device=device, dtype=torch.float64)
        self.sumy2 = torch.zeros(bands, device=device, dtype=torch.float64)
        self.sumxy = torch.zeros(bands, device=device, dtype=torch.float64)

    @torch.no_grad()
    def add(self, yhat: torch.Tensor, y: torch.Tensor, m: torch.Tensor):
        B, C, H, W = y.shape
        assert C == self.bands
        m_bool = (m > 0.5)
        v = m_bool.view(B, C, -1)
        nx = v.sum(dim=2).to(torch.float64)
        self.n += nx.sum(dim=0)
        x = (yhat * m).view(B, C, -1).to(torch.float64)
        t = (y * m).view(B, C, -1).to(torch.float64)
        self.sumx += x.sum(dim=2).sum(dim=0)
        self.sumy += t.sum(dim=2).sum(dim=0)
        self.sumx2 += (x*x).sum(dim=2).sum(dim=0)
        self.sumy2 += (t*t).sum(dim=2).sum(dim=0)
        self.sumxy += (x*t).sum(dim=2).sum(dim=0)

    def compute(self) -> Dict[str, Any]:
        eps = 1e-12
        n = self.n.clamp(min=1.0)
        num = self.sumxy - (self.sumx * self.sumy / n)
        denx = self.sumx2 - (self.sumx * self.sumx / n)
        deny = self.sumy2 - (self.sumy * self.sumy / n)
        r = num / (torch.sqrt(denx.clamp(min=eps)) * torch.sqrt(deny.clamp(min=eps)) + eps)
        return {"pearson_r_per_band": r.double().cpu().numpy().tolist(),
                "n_per_band": self.n.double().cpu().numpy().tolist()}


In [66]:
@torch.no_grad()
def _hotspot_metrics(yhat: torch.Tensor, y: torch.Tensor, m: torch.Tensor, top_q: float = 0.75) -> Dict[str, Any]:
    B, C, H, W = y.shape
    yh, gt, ms = yhat.view(B, C, -1), y.view(B, C, -1), m.view(B, C, -1)
    overall_acc = {"precision": 0.0, "recall": 0.0, "f1": 0.0, "iou": 0.0}
    overall_n = 0
    pb = {k: torch.zeros(C, dtype=torch.float64) for k in overall_acc}
    pb['n'] = torch.zeros(C, dtype=torch.int64)

    for b in range(B):
        for c in range(C):
            valid = ms[b,c] > 0.5
            if valid.sum() < 10: continue
            gt_valid, yh_valid = gt[b,c][valid], yh[b,c][valid]
            thr = torch.quantile(gt_valid, top_q)
            gt_bin, yh_bin = (gt_valid >= thr), (yh_valid >= thr)
            tp = (yh_bin & gt_bin).sum().item()
            fp = (yh_bin & ~gt_bin).sum().item()
            fn = (~yh_bin & gt_bin).sum().item()
            prec = tp / (tp + fp + 1e-9)
            rec = tp / (tp + fn + 1e-9)
            f1 = 2 * prec * rec / (prec + rec + 1e-9)
            iou = tp / ((yh_bin | gt_bin).sum().item() + 1e-9)
            for k, v in zip(overall_acc.keys(), [prec, rec, f1, iou]):
                overall_acc[k] += v
                pb[k][c] += v
            overall_n += 1
            pb['n'][c] += 1

    result_overall = {k: v / overall_n if overall_n > 0 else 0.0 for k, v in overall_acc.items()}
    result_overall['n'] = overall_n
    per_band = {k: (pb[k] / pb['n'].clamp(min=1).to(torch.float64)).tolist() for k in overall_acc}
    per_band['n'] = pb['n'].tolist()
    return {"overall": result_overall, "per_band": per_band}


In [67]:
# ---------- Loader builders for LOCO / temporal ----------
import torch
from typing import List, Optional, Tuple

def _year_from_t(meta_years: Optional[List[int]], t: int, base_default: int = 2003) -> int:
    base = meta_years[0] if meta_years else base_default
    return base + (t // 12)

def _build_index(ds, include_cities: List[str], year_range: Optional[Tuple[int,int]] = None) -> List[int]:
    idxs = []
    for i, (city, t) in enumerate(ds.indices):
        if city not in include_cities:
            continue
        meta_years = ds.data[city].get('meta', {}).get('years', [])
        y = _year_from_t(meta_years, t)
        if (year_range is None) or (year_range[0] <= y <= year_range[1]):
            idxs.append(i)
    return idxs

def _resolve_loader_config(ds, num_workers: int, pin_memory: bool):
    data_device = getattr(ds, 'device', torch.device('cpu'))
    if not isinstance(data_device, torch.device):
        data_device = torch.device(data_device)
    use_gpu = data_device.type != 'cpu'
    worker_count = 0 if use_gpu else num_workers
    persistent_workers = bool(worker_count)
    effective_pin = pin_memory and not use_gpu
    return worker_count, persistent_workers, effective_pin

def make_loaders_loco(ds, heldout_city: str, batch_size: int = 8, num_workers: int = 4,
                      val_year: int = 2019, pin_memory: bool = True):
    cities = sorted({c for c, _ in ds.indices})
    train_cities = [c for c in cities if c != heldout_city]

    worker_count, persistent_workers, effective_pin = _resolve_loader_config(ds, num_workers, pin_memory)

    train_idxs_all = _build_index(ds, train_cities, None)
    val_idxs = _build_index(ds, train_cities, (val_year, val_year))
    val_set = set(val_idxs)
    train_idxs = [i for i in train_idxs_all if i not in val_set]
    test_idxs = _build_index(ds, [heldout_city], None)

    train_loader = DataLoader(Subset(ds, train_idxs), batch_size=batch_size, shuffle=True,
                              num_workers=worker_count, pin_memory=effective_pin,
                              persistent_workers=persistent_workers)
    val_loader = (DataLoader(Subset(ds, val_idxs), batch_size=batch_size, shuffle=False,
                             num_workers=worker_count, pin_memory=effective_pin,
                             persistent_workers=persistent_workers)
                  if val_idxs else None)
    test_loader = DataLoader(Subset(ds, test_idxs), batch_size=batch_size, shuffle=False,
                             num_workers=worker_count, pin_memory=effective_pin,
                             persistent_workers=persistent_workers)
    info = {"train_size": len(train_idxs), "val_size": len(val_idxs), "test_size": len(test_idxs),
            "train_cities": train_cities, "heldout_city": heldout_city}
    return train_loader, val_loader, test_loader, info


def make_loaders_temporal(ds, batch_size: int = 8, num_workers: int = 4,
                          train_years: Tuple[int, int] = (2003, 2018),
                          val_year: int = 2019, test_year: int = 2020,
                          include_cities: Optional[List[str]] = None,
                          pin_memory: bool = True):
    all_cities = sorted({c for c, _ in ds.indices})
    cities = include_cities or all_cities
    worker_count, persistent_workers, effective_pin = _resolve_loader_config(ds, num_workers, pin_memory)

    train_idxs = _build_index(ds, cities, train_years)
    val_idxs = _build_index(ds, cities, (val_year, val_year))
    test_idxs = _build_index(ds, cities, (test_year, test_year))

    train_loader = DataLoader(Subset(ds, train_idxs), batch_size=batch_size, shuffle=True,
                              num_workers=worker_count, pin_memory=effective_pin,
                              persistent_workers=persistent_workers)
    val_loader = DataLoader(Subset(ds, val_idxs), batch_size=batch_size, shuffle=False,
                            num_workers=worker_count, pin_memory=effective_pin,
                            persistent_workers=persistent_workers)
    test_loader = DataLoader(Subset(ds, test_idxs), batch_size=batch_size, shuffle=False,
                             num_workers=worker_count, pin_memory=effective_pin,
                             persistent_workers=persistent_workers)
    info = {"train_years": train_years, "val_year": val_year, "test_year": test_year, "cities": cities}
    return train_loader, val_loader, test_loader, info


In [68]:
import os, json, csv
from collections import defaultdict
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate_best(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    device: torch.device,
    save_dir: str,
    compute_alpha_hist: bool = True,
    alpha_hist_bins: int = 20,
    band_names: list[str] | None = None,
):
    """
    Evaluates a trained model on `loader`, saves a suite of metrics/plots to `save_dir`,
    and returns a summary dict.

    Expects each batch to be a dict with:
      "pm", "pm_mask", "lst", "lst_mask", "lc", "uhii", "uhii_mask", "city"
    and model(pm=..., pm_mask=..., lst=..., lst_mask=..., lc=..., return_alpha=...) -> {"yhat": ..., "alpha": ...?}
    """
    os.makedirs(save_dir, exist_ok=True)
    model.eval()

    # Lazy alloc once we know C
    C = None
    # Overall scalars (averaged over batches)
    overall = defaultdict(float)
    n_batches = 0

    # Per-band accumulators (lazy)
    sse_c = y_sum_c = y2_sum_c = m_sum_c = None
    mae_sum_c = mae_den_c = None
    ssim_sum_c = None
    ssim_n_c = 0.0
    corr_accum = None

    # Per-city accumulators
    per_city_stats = defaultdict(lambda: defaultdict(float))  # keys: w, mae, rmse, ssim, and per-band num/den

    # Hotspot accumulators
    hotspot_overall_acc = defaultdict(float)  # precision, recall, f1, iou (weighted by n)
    hotspot_overall_n = 0.0
    hotspot_pb_sum = hotspot_pb_n = None

    # Stream-attention α collection
    alpha_vals = []

    for batch in tqdm(loader, desc="Evaluating test set"):
        pm, pm_m   = batch["pm"].to(device),      batch["pm_mask"].to(device)
        lst, lst_m = batch["lst"].to(device),     batch["lst_mask"].to(device)
        lc         = batch["lc"].to(device).long()
        y, m       = batch["uhii"].to(device),    batch["uhii_mask"].to(device)
        cities     = batch["city"]  # keep on CPU

        if C is None:
            C = y.shape[1]
            # Per-band tensors (double on CPU for numeric stability)
            sse_c     = torch.zeros(C, dtype=torch.float64)
            y_sum_c   = torch.zeros(C, dtype=torch.float64)
            y2_sum_c  = torch.zeros(C, dtype=torch.float64)
            m_sum_c   = torch.zeros(C, dtype=torch.float64)
            mae_sum_c = torch.zeros(C, dtype=torch.float64)
            mae_den_c = torch.zeros(C, dtype=torch.float64)
            ssim_sum_c= torch.zeros(C, dtype=torch.float64)
            corr_accum = CorrAccumulator(bands=C, device=device)
            if band_names is None or len(band_names) != C:
                band_names = [f"band_{i}" for i in range(C)]
            hotspot_pb_sum = {k: torch.zeros(C, dtype=torch.float64) for k in ("precision","recall","f1","iou")}
            hotspot_pb_n   = torch.zeros(C, dtype=torch.float64)

        # Forward
        out = model(pm=pm, pm_mask=pm_m, lst=lst, lst_mask=lst_m, lc=lc, return_alpha=compute_alpha_hist)
        yhat = out["yhat"]

        # Overall masked metrics
        overall["mae"]  += masked_mae(yhat, y, m).item()
        overall["rmse"] += masked_rmse(yhat, y, m).item()
        overall["ssim"] += masked_ssim(yhat, y, m).item()
        n_batches += 1

        # Per-band sums
        sse_c     += ((yhat - y) ** 2 * m).sum(dim=(0, 2, 3)).double().cpu()
        y_sum_c   += (y * m).sum(dim=(0, 2, 3)).double().cpu()
        y2_sum_c  += (y * y * m).sum(dim=(0, 2, 3)).double().cpu()
        m_sum_c   += m.sum(dim=(0, 2, 3)).double().cpu()
        mae_sum_c += (torch.abs(yhat - y) * m).sum(dim=(0, 2, 3)).double().cpu()
        mae_den_c += m.sum(dim=(0, 2, 3)).double().cpu()

        # Per-band SSIM (compute on each channel to honor masks)
        ssim_vals = []
        for ci in range(C):
            ssim_vals.append(masked_ssim(yhat[:, ci:ci+1], y[:, ci:ci+1], m[:, ci:ci+1]).item())
        ssim_sum_c += torch.tensor(ssim_vals, dtype=torch.float64)
        ssim_n_c   += 1.0

        # Pearson r (streaming)
        corr_accum.add(yhat, y, m)

        # Hotspot metrics (overall + per-band, weighted by per-band n)
        h_res = _hotspot_metrics(yhat, y, m, top_q=0.75)
        n_h = h_res["overall"]["n"]
        if n_h > 0:
            for k in ("precision","recall","f1","iou"):
                hotspot_overall_acc[k] += h_res["overall"][k] * n_h
            hotspot_overall_n += n_h
        pb_n_tensor = torch.tensor(h_res["per_band"]["n"], dtype=torch.float64)
        for k in ("precision","recall","f1","iou"):
            hotspot_pb_sum[k] += torch.tensor(h_res["per_band"][k], dtype=torch.float64) * pb_n_tensor
        hotspot_pb_n += pb_n_tensor

        # Per-city metrics (mask-weighted)
        for i, city in enumerate(cities):
            w = m[i].sum().item()
            if w <= 0:
                continue
            per_city_stats[city]["w"]    += w
            per_city_stats[city]["mae"]  += masked_mae (yhat[i:i+1], y[i:i+1], m[i:i+1]).item() * w
            per_city_stats[city]["rmse"] += masked_rmse(yhat[i:i+1], y[i:i+1], m[i:i+1]).item() * w
            per_city_stats[city]["ssim"] += masked_ssim(yhat[i:i+1], y[i:i+1], m[i:i+1]).item() * w

            # per-band MAE per city
            if "per_band_num" not in per_city_stats[city]:
                per_city_stats[city]["per_band_num"] = torch.zeros(C, dtype=torch.float64)
                per_city_stats[city]["per_band_den"] = torch.zeros(C, dtype=torch.float64)
            per_city_stats[city]["per_band_num"] += (torch.abs(yhat[i] - y[i]) * m[i]).sum(dim=(1,2)).double().cpu()
            per_city_stats[city]["per_band_den"] += m[i].sum(dim=(1,2)).double().cpu()

        # Collect stream attention weights
        if compute_alpha_hist and ("alpha" in out) and (out["alpha"] is not None):
            alpha_vals.append(out["alpha"].detach().float().cpu().numpy())

    if C is None:
        raise RuntimeError("evaluate_best: empty loader (no batches).")

    # Finalize overall averages
    for k in ("mae","rmse","ssim"):
        overall[k] /= max(1, n_batches)

    # Finalize per-band stats
    sst_c = y2_sum_c - (y_sum_c * y_sum_c) / m_sum_c.clamp(min=1.0)  # exact SST around global mean
    mae_per_band  = (mae_sum_c / mae_den_c.clamp(min=1.0)).numpy().tolist()
    rmse_per_band = torch.sqrt(sse_c / m_sum_c.clamp(min=1.0)).numpy().tolist()
    ssim_per_band = (ssim_sum_c / max(1.0, ssim_n_c)).numpy().tolist()
    r2_per_band   = (1.0 - (sse_c / sst_c.clamp(min=1e-9))).numpy().tolist()

    overall["r2"] = 1.0 - (sse_c.sum().item() / sst_c.sum().clamp(min=1e-9).item())

    corr = corr_accum.compute()
    r_per_band = corr["pearson_r_per_band"]
    n_per_band = corr.get("n_per_band", [])

    # Hotspot finalize
    hotspot_overall = {k: (hotspot_overall_acc[k] / max(1.0, hotspot_overall_n)) for k in ("precision","recall","f1","iou")}
    hotspot_per_band = {k: (hotspot_pb_sum[k] / hotspot_pb_n.clamp(min=1.0)).numpy().tolist() for k in ("precision","recall","f1","iou")}

    # ---- Save artifacts ----
    # 1) Overall JSON
    with open(os.path.join(save_dir, "test_overall.json"), "w") as f:
        json.dump(
            {
                "overall": overall,
                "pearson_r_per_band": r_per_band,
                "n_per_band": n_per_band,
                "hotspot": hotspot_overall,
                "hotspot_per_band": hotspot_per_band,
            },
            f, indent=2
        )

    # 2) Per-city CSV (mask-weighted)
    per_city_rows = []
    for city, stats in per_city_stats.items():
        w = max(1.0, stats["w"])
        per_city_rows.append({
            "city": city,
            "mae":  stats["mae"]  / w,
            "rmse": stats["rmse"] / w,
            "ssim": stats["ssim"] / w,
        })
    per_city_rows = sorted(per_city_rows, key=lambda r: r["city"])
    with open(os.path.join(save_dir, "test_per_city.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["city", "mae", "rmse", "ssim"])
        writer.writeheader()
        writer.writerows(per_city_rows)

    # 3) Overall per-band CSV
    with open(os.path.join(save_dir, "test_overall_per_band.csv"), "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["band", "MAE", "RMSE", "SSIM", "R2", "Pearson_r"])
        for i, bname in enumerate(band_names):
            writer.writerow([bname, mae_per_band[i], rmse_per_band[i], ssim_per_band[i], r2_per_band[i], r_per_band[i]])

    # 4) Per-city per-band MAE CSV
    header = ["city"] + [f"MAE_{b}" for b in band_names] + ["MAE_overall"]
    per_city_band_rows = []
    for city, stats in per_city_stats.items():
        num = stats["per_band_num"]
        den = stats["per_band_den"].clamp(min=1.0)
        mae_c = (num / den).numpy().tolist()
        row = {"city": city}
        for i, bname in enumerate(band_names):
            row[f"MAE_{bname}"] = mae_c[i]
        row["MAE_overall"] = next((r["mae"] for r in per_city_rows if r["city"] == city), 0.0)
        per_city_band_rows.append(row)

    with open(os.path.join(save_dir, "test_per_city_per_band.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        writer.writerows(sorted(per_city_band_rows, key=lambda r: r["city"]))

    # 5) Stream-attention α plots (optional)
    if compute_alpha_hist and len(alpha_vals) > 0:
        A = np.concatenate(alpha_vals, axis=0)  # (N, 3)
        plt.figure(figsize=(6, 4))
        labels_alpha = ["LC", "LST", "PM25"]
        for i in range(min(A.shape[1], 3)):
            plt.hist(A[:, i], bins=alpha_hist_bins, alpha=0.6, label=labels_alpha[i] if i < len(labels_alpha) else f"a{i}")
        plt.legend()
        plt.title("Stream Attention α Weights")
        plt.xlabel("Attention Weight")
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "alpha_hist.png"), dpi=130)
        plt.close()

        with open(os.path.join(save_dir, "alpha_mean.json"), "w") as f:
            mu = A.mean(axis=0).tolist()
            json.dump({"alpha_mean": { (labels_alpha[i] if i < len(labels_alpha) else f"a{i}"): mu[i] for i in range(len(mu)) }},
                      f, indent=2)

    final_results = {
        "overall": overall,
        "hotspot": hotspot_overall,
        "per_band": {
            "mae": mae_per_band,
            "rmse": rmse_per_band,
            "ssim": ssim_per_band,
            "r2": r2_per_band,
            "pearson_r": r_per_band,
        },
        "per_city": per_city_rows,
    }
    print(f"✅ Evaluation complete. Results saved to: {save_dir}")
    return final_results


In [69]:
# ============================
# INTERPRETABILITY: LC CBAM MAPS
# ============================
import os, json
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

@torch.no_grad()
def _cbam_channel_att(module, x):
    """
    Recompute CBAM channel attention (before spatial) using module's layers.
    Returns (B, C, 1, 1) in [0,1].
    """
    B, C, H, W = x.shape
    avg = module.avg_pool(x).view(B, C)   # (B,C)
    mx  = module.max_pool(x).view(B, C)   # (B,C)
    ca  = module.mlp(avg) + module.mlp(mx)  # (B,C)
    ca  = module.sigmoid(ca).view(B, C, 1, 1)
    return ca

@torch.no_grad()
def _cbam_spatial_att(module, x_after_channel):
    """
    Recompute CBAM spatial attention (after channel weighting) using module's layers.
    Returns (B,1,H,W) in [0,1].
    """
    avg_sp = torch.mean(x_after_channel, dim=1, keepdim=True)
    max_sp, _ = torch.max(x_after_channel, dim=1, keepdim=True)
    spatial_in = torch.cat([avg_sp, max_sp], dim=1)
    sa = torch.sigmoid(module.spatial_conv(spatial_in))
    return sa

@torch.no_grad()
def dump_lc_cbam_maps(model, loader, device, save_dir: str, max_samples: int = 6):
    """
    Saves:
      - Spatial attention maps from LCEncoder CBAM blocks (PNG)
      - Channel attention vectors (CSV + barplot PNG)
    Looks for modules named: 'lc_encoder.block1.cbam', 'lc_encoder.block2.cbam'.
    """
    os.makedirs(save_dir, exist_ok=True)
    # Find LC CBAMs
    targets = []
    for name, mod in model.named_modules():
        if name.endswith("lc_encoder.block1.cbam") or name.endswith("lc_encoder.block2.cbam"):
            targets.append((name, mod))
    if not targets:
        print("No LC CBAM modules found (expected lc_encoder.block1.cbam / block2.cbam).")
        return

    saved = 0
    rows_csv = []  # each row: {"name": ..., "sample_id": ..., "channel_att":[...]}
    sample_id = 0

    for batch in loader:
        if saved >= max_samples: break
        # inputs required only to run the model through LC path
        pm, pm_m   = batch["pm"].to(device),      batch["pm_mask"].to(device)
        lst, lst_m = batch["lst"].to(device),     batch["lst_mask"].to(device)
        lc         = batch["lc"].to(device).long()
        cities     = batch["city"]

        # Forward once to land tensors on device (we'll recompute attentions via modules)
        _ = model(pm=pm, pm_mask=pm_m, lst=lst, lst_mask=lst_m, lc=lc)

        # We must get the LC feature **before** patchify (i.e., the 1333/1320 grid) — but your LCEncoder
        # encapsulates that. So we recompute inside each CBAM: we hook at their inputs by registering
        # a forward_pre_hook temporarily to capture x entering CBAM.
        captured_inputs = {}

        def make_pre(name):
            def _pre_hook(m, inp):
                # inp is a tuple with x
                captured_inputs[name] = inp[0].detach()  # (B,C,H,W)
            return _pre_hook

        handles = []
        for name, mod in targets:
            handles.append(mod.register_forward_pre_hook(make_pre(name)))

        # re-run just the LC path by calling the full model (cheap enough at 40x40 stage)
        _ = model(pm=pm, pm_mask=pm_m, lst=lst, lst_mask=lst_m, lc=lc)

        # Now for each target, recompute attentions from captured inputs and save
        for (name, mod) in targets:
            if name not in captured_inputs:
                continue
            x_in = captured_inputs[name]          # (B,C,40,40)
            ca = _cbam_channel_att(mod, x_in)     # (B,C,1,1)
            x_ch = x_in * ca                      # channel-weighted
            sa = _cbam_spatial_att(mod, x_ch)     # (B,1,40,40)

            B, C, H, W = x_in.shape
            n_take = min(B, max_samples - saved)

            # Save spatial heatmaps
            for i in range(n_take):
                plt.figure(figsize=(3,3))
                plt.imshow(sa[i,0].cpu().numpy(), cmap="magma", interpolation="nearest")
                plt.title(f"{name} spatial (sample {sample_id+i})")
                plt.axis("off"); plt.tight_layout()
                plt.savefig(os.path.join(save_dir, f"cbam_spatial_{name.replace('.','_')}_s{sample_id+i}.png"), dpi=140)
                plt.close()

            # Save channel attentions: CSV + bar plot for the first 1-2
            for i in range(n_take):
                vec = ca[i,:,0,0].cpu().numpy()
                rows_csv.append({
                    "module": name,
                    "sample_id": sample_id + i,
                    "city": str(cities[i]),
                    "channel_att": vec.tolist()
                })
                if i < 2:  # keep plots light
                    plt.figure(figsize=(6,2.2))
                    plt.bar(np.arange(C), vec)
                    plt.ylim(0,1)
                    plt.title(f"{name} channel α (sample {sample_id+i})")
                    plt.tight_layout()
                    plt.savefig(os.path.join(save_dir, f"cbam_channel_{name.replace('.','_')}_s{sample_id+i}.png"), dpi=140)
                    plt.close()

            saved += n_take
            sample_id += n_take
            if saved >= max_samples:
                break

        for h in handles:
            h.remove()

    # Write channel attention CSV (module, sample_id, city, C values)
    if rows_csv:
        import csv
        # wide CSV with columns: module, sample_id, city, ch_0..ch_{C-1}
        maxC = max(len(r["channel_att"]) for r in rows_csv)
        cols = ["module","sample_id","city"] + [f"ch_{i}" for i in range(maxC)]
        csv_path = os.path.join(save_dir, "cbam_channel_attention.csv")
        with open(csv_path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(cols)
            for r in rows_csv:
                row = [r["module"], r["sample_id"], r["city"]] + r["channel_att"]
                w.writerow(row)
        print(f"CBAM attention maps saved to: {save_dir}")


In [70]:
# ============================
# INTERPRETABILITY: U-Net AttentionGate Masks
# ============================
@torch.no_grad()
def _gate_mask(module, x, g):
    """
    Recompute AttentionGate mask = sigmoid(psi(ReLU(theta_x(x) + phi_g(g_up))))
    Upsamples g to x spatial size when needed (matches your AttentionGate forward).
    Returns (B,1,H,W) in [0,1].
    """
    if g.shape[-2:] != x.shape[-2:]:
        g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)
    attn = F.relu(module.theta_x(x) + module.phi_g(g), inplace=False)
    attn = torch.sigmoid(module.psi(attn))
    return attn

@torch.no_grad()
def dump_unet_gate_masks(model, loader, device, save_dir: str, max_samples: int = 4):
    """
    Saves AttentionGate masks for a few test tiles.
    Looks for gates: model.unet.g2 (depth=4), model.unet.g1_d4, model.unet.g1_d3 (depth=3), model.unet.g0
    We recompute masks from inputs captured via forward hooks at the right points.
    """
    os.makedirs(save_dir, exist_ok=True)
    model.eval()

    # Hooks to capture intermediate activations needed for gates
    feats = {}
    handles = []
    unet = model.unet

    def stash(name):
        def _hook(m, inp, out): feats[name] = out.detach()
        return _hook

    handles.append(unet.enc0.register_forward_hook(stash("e0")))
    handles.append(unet.enc1.register_forward_hook(stash("e1")))
    if hasattr(unet, "bot"): handles.append(unet.bot.register_forward_hook(stash("bottleneck")))
    if hasattr(unet, "dec2"): handles.append(unet.dec2.register_forward_hook(stash("d2")))

    saved_samples = 0
    for batch in loader:
        if saved_samples >= max_samples: break
        # Run model to trigger hooks
        _ = model(
            pm=batch["pm"].to(device), pm_mask=batch["pm_mask"].to(device),
            lst=batch["lst"].to(device), lst_mask=batch["lst_mask"].to(device),
            lc=batch["lc"].to(device).long()
        )

        # Recompute and save gate masks
        gate_entries = []
        if hasattr(unet, "g0") and "e0" in feats and "d2" in feats:
            gate_entries.append(("g0", unet.g0, feats["e0"], feats["d2"]))
        if hasattr(unet, "g1_d3") and "e1" in feats and "bottleneck" in feats: # For depth=3
            gate_entries.append(("g1", unet.g1_d3, feats["e1"], feats["bottleneck"]))

        for name, module, x, g in gate_entries:
            mask = _gate_mask(module, x, g)
            for i in range(min(mask.shape[0], max_samples - saved_samples)):
                plt.figure(figsize=(3, 3))
                plt.imshow(mask[i, 0].cpu().numpy(), cmap="viridis")
                plt.axis("off")
                plt.title(f"Gate {name} - Sample {saved_samples + i}")
                plt.savefig(os.path.join(save_dir, f"gate_{name}_sample_{saved_samples + i}.png"))
                plt.close()
        saved_samples += batch['lc'].shape[0]

    for h in handles: h.remove()
    print(f"✅ U-Net attention gate masks saved to {save_dir}")

In [71]:
def dump_cbam_spatial_maps(model, loader, device, save_dir: str, max_samples: int = 4):
    """
    Registers forward hooks on LCEncoder's CBAM modules to dump spatial attention maps at 40×40.
    Saves a few PNGs for qualitative inspection.
    """
    os.makedirs(save_dir, exist_ok=True)
    # find CBAM modules (assuming LCEncoder.block1.cbam, block2.cbam)
    cbams = []
    for name, module in model.named_modules():
        if name.endswith("block1.cbam") or name.endswith("block2.cbam"):
            cbams.append((name, module))
    if not cbams:
        print("No CBAM modules found for visualization.")
        return

    captured = {}

    def make_hook(name):
        def _hook(module, inp, out):
            # out is post spatial attention already; we want the spatial map → re-derive via module.spatial_conv(sigmoid pre)
            # Approx: compute from out & input is not trivial; Instead, grab spatial map by re-applying spatial branch:
            x = out.detach()  # (B,C,H,W)
            # Spatial attention was applied; not recoverable exactly. We'll proxy by averaging channels to visualize.
            att_sp = x.mean(dim=1, keepdim=True)  # proxy heat
            captured[name] = att_sp.cpu()
        return _hook

    handles = [m.register_forward_hook(make_hook(n)) for n,m in cbams]

    saved = 0
    with torch.no_grad():
        for batch in loader:
            pm, pm_m   = batch["pm"].to(device),        batch["pm_mask"].to(device)
            lst, lst_m = batch["lst"].to(device),       batch["lst_mask"].to(device)
            lc         = batch["lc"].to(device).long()
            # one forward to trigger hooks
            _ = model(pm=pm, pm_mask=pm_m, lst=lst, lst_mask=lst_m, lc=lc)
            # dump a few maps
            for name, att in captured.items():
                for i in range(min(att.shape[0], max_samples - saved)):
                    plt.figure(figsize=(3,3))
                    plt.imshow(att[i,0].numpy(), cmap="magma")
                    plt.axis("off"); plt.title(name)
                    plt.tight_layout()
                    plt.savefig(os.path.join(save_dir, f"cbam_{name.replace('.', '_')}_{saved:02d}.png"), dpi=130)
                    plt.close()
                    saved += 1
                    if saved >= max_samples:
                        break
            if saved >= max_samples:
                break

    for h in handles:
        h.remove()

In [72]:
# ---------- Ablation grid & LOCO orchestrator ----------

def _cfg_variant(base: "SystemConfig",
                 name: str,
                 **overrides) -> Tuple[str, "SystemConfig"]:
    cfg = replace(base, **overrides)
    return name, cfg

def default_ablation_grid(base: "SystemConfig") -> List[Tuple[str, "SystemConfig"]]:
    """
    5-row minimal grid:
      1) Plain UNet, depth=3, LC-CBAM on (you can later swap to LC-Fractions by swapping lc_encoder)
      2) Plain UNet, depth=4, LC-CBAM
      3) Att-UNet, depth=4, LC-CBAM  <-- main
      4) (3) -LC
      5) (3) -PM2.5
    """
    rows = []
    rows.append(_cfg_variant(base, "plain_d3", use_unet_attn=False, depth=3, use_stream_attn=False))
    rows.append(_cfg_variant(base, "plain_d4", use_unet_attn=False, depth=4, use_stream_attn=False))
    rows.append(_cfg_variant(base, "att_d4_main", use_unet_attn=True, depth=4, use_stream_attn=True))
    rows.append(_cfg_variant(base, "att_d4_noLC", use_unet_attn=True, depth=4, use_stream_attn=True, use_lc=False))
    rows.append(_cfg_variant(base, "att_d4_noPM", use_unet_attn=True, depth=4, use_stream_attn=True, use_pm=False))
    return rows



In [73]:

# ===================================================================
# PART 3: OPTIMIZED run_loco_grid FUNCTION
# ===================================================================
import pandas as pd

def run_loco_grid(
    root_dir: str,
    ds,
    base_cfg: "SystemConfig",
    build_system_fn,
    fit_fn,
    epochs: int = 25,
    batch_size: int = 8,
    num_workers: int = 4,
    lambda_ssim: float = 0.2,
    early_stop_patience: int = 10,
    runs_root: str = "runs",
    folds: list[str] | None = None
):
    import pandas as pd
    os.makedirs(runs_root, exist_ok=True)

    # If folds not provided, or provided as ["ALL"], run LOCO on all cities in dataset
    all_cities = sorted({c for c,_ in ds.indices})
    folds_to_run = all_cities if (not folds or (len(folds) == 1 and folds[0].upper() == "ALL")) else folds

    summary_rows = []
    print(f"Starting LOCO evaluation for {len(folds_to_run)} folds: {folds_to_run}")

    for heldout_city in folds_to_run:
        print(f"\n{'='*25} FOLD: Holding out {heldout_city.upper()} {'='*25}")
        fold_dir = os.path.join(runs_root, f"LOCO_{heldout_city}")
        os.makedirs(fold_dir, exist_ok=True)

        # Build loaders for this fold
        train_loader, val_loader, test_loader, info = make_loaders_loco(
            ds, heldout_city=heldout_city, batch_size=batch_size,
            num_workers=num_workers, val_year=2019
        )
        if val_loader is None or len(val_loader) == 0:
            print("Warning: Validation loader is empty. Early stopping will not be effective.")

        # Steps, config, system
        steps_per_epoch = max(1, math.ceil(len(train_loader.dataset) / batch_size))
        total_steps = epochs * steps_per_epoch
        cfg_run = replace(base_cfg, total_steps=total_steps, warmup_steps=steps_per_epoch)
        system = build_system_fn(root_dir, cfg_run, cities_priority=info["train_cities"])

        # Train
        print(f"Training on {info['train_size']} samples, validating on {info['val_size']}...")
        fit_fn(
            system, train_loader, val_loader, fold_dir,
            epochs=epochs, lambda_ssim=lambda_ssim, early_stop_patience=early_stop_patience
        )

        # Load best and evaluate
        print(f"--- Evaluating best model on held-out city: {heldout_city} ---")
        best_ckpt_path = os.path.join(fold_dir, "best.pt")
        if not os.path.exists(best_ckpt_path):
            print(f"Warning: Could not find 'best.pt' for fold {heldout_city}. Skipping evaluation.")
            continue

        ckpt = torch.load(best_ckpt_path, map_location=system["device"])
        system["model"].load_state_dict(ckpt["model"])

        # Read UHII band names from held-out city's meta (if available)
        band_names = None
        meta_p = os.path.join(root_dir, heldout_city, "meta.json")
        if os.path.exists(meta_p):
            try:
                with open(meta_p, "r") as f:
                    meta = json.load(f)
                uh_order = meta.get("uhii_bands_order", [])
                if uh_order:
                    band_names = uh_order
            except Exception as e:
                print(f"Warning: Could not read band names for {heldout_city}: {e}")

        test_results = evaluate_best(
            system["model"], test_loader, system["device"], save_dir=fold_dir,
            compute_alpha_hist=True, band_names=band_names
        )

        interp_root = os.path.join(fold_dir, "interpretability")
        cbam_dir    = os.path.join(interp_root, "cbam")
        gates_dir   = os.path.join(interp_root, "gates")
        dump_lc_cbam_maps(system["model"], test_loader, system["device"], save_dir=cbam_dir,  max_samples=6)
        dump_unet_gate_masks(system["model"], test_loader, system["device"], save_dir=gates_dir, max_samples=6)

        dump_cbam_spatial_maps(
            system["model"], test_loader, system["device"],
            save_dir=os.path.join(fold_dir, "cbam_maps"), max_samples=4
        )

        row = {
            "fold": heldout_city,
            "test_mae": test_results["overall"]["mae"],
            "test_rmse": test_results["overall"]["rmse"],
            "test_ssim": test_results["overall"]["ssim"],
            "test_r2": test_results["overall"]["r2"],
            "hotspot_f1": test_results["hotspot"]["f1"],
            "hotspot_iou": test_results["hotspot"]["iou"],
        }
        summary_rows.append(row)
        print(f"Results for fold {heldout_city}: MAE={row['test_mae']:.4f}, R2={row['test_r2']:.4f}")

    if not summary_rows:
        print("\nNo folds were completed. No summary report generated.")
        return

    # Single combined summary across all folds
    summary_df = pd.DataFrame(summary_rows)
    mean_row = summary_df.mean(numeric_only=True)
    std_row  = summary_df.std(numeric_only=True)
    mean_row['fold'] = 'mean'
    std_row['fold']  = 'std'
    summary_df = pd.concat([summary_df, mean_row.to_frame().T, std_row.to_frame().T], ignore_index=True)

    summary_path = os.path.join(runs_root, "summary_loco.csv")
    summary_df.to_csv(summary_path, index=False, float_format='%.4f')
    print(f"\n{'='*25} LOCO EVALUATION COMPLETE {'='*25}")
    print("Final Summary across all folds:")
    print(summary_df.to_string(index=False))
    print(f"\n✅ Summary report saved to: {summary_path}")

In [74]:
from dataclasses import replace

def full_ablation_grid(base: "SystemConfig"):
    """
    Minimal but complete ablation set:
      - Plain U-Net vs Attention U-Net
      - Depth 3 vs 4
      - Stream removals: -LC, -PM, -LST
      - LC path: CBAM vs Fractions
      - Stream-attn on/off
    """
    rows = []
    add = rows.append

    # Baselines (plain UNet, no stream-attn)
    add(("plain_d3_cbam",
         replace(base, use_unet_attn=False, depth=3, use_stream_attn=False, lc_encoder_type="cbam")))
    add(("plain_d4_cbam",
         replace(base, use_unet_attn=False, depth=4, use_stream_attn=False, lc_encoder_type="cbam")))

    # Main (attention UNet + stream-attn)
    add(("att_d4_cbam_main",
         replace(base, use_unet_attn=True, depth=4, use_stream_attn=True, lc_encoder_type="cbam")))
    add(("att_d3_cbam_main",
         replace(base, use_unet_attn=True, depth=3, use_stream_attn=True, lc_encoder_type="cbam")))

    # Remove streams (on top of main)
    add(("att_d4_cbam_noLC",
         replace(base, use_unet_attn=True, depth=4, use_stream_attn=True, lc_encoder_type="cbam", use_lc=False)))
    add(("att_d4_cbam_noPM",
         replace(base, use_unet_attn=True, depth=4, use_stream_attn=True, lc_encoder_type="cbam", use_pm=False)))
    add(("att_d4_cbam_noLST",
         replace(base, use_unet_attn=True, depth=4, use_stream_attn=True, lc_encoder_type="cbam", use_lst=False)))

    # LC Fractions path
    add(("att_d4_fracLC_main",
         replace(base, use_unet_attn=True, depth=4, use_stream_attn=True, lc_encoder_type="frac")))

    # Stream attention OFF (keep attention-UNet)
    add(("att_d4_cbam_noStreamAttn",
         replace(base, use_unet_attn=True, depth=4, use_stream_attn=False, lc_encoder_type="cbam")))

    return rows


In [79]:
import os
import pandas as pd

def run_all_ablation_loco(
    root_dir: str,
    ds,
    base_cfg: SystemConfig,
    build_system_fn,
    fit_fn,
    cities: list[str] | None = None,
    epochs: int = 25,
    batch_size: int = 32,
    num_workers: int = 2,
    lambda_ssim: float = 0.2,
    early_stop_patience: int = 10,
    runs_root: str = "runs",
    folds: list[str] | None = None  # None => use provided cities or dataset
):
    os.makedirs(runs_root, exist_ok=True)
    grid = full_ablation_grid(base_cfg)
    combined_rows = []
    folds_to_use = folds if folds is not None else (list(cities) if cities else None)

    for variant_name, cfg in grid:
        print(f"\n##### VARIANT: {variant_name} #####")
        variant_dir = os.path.join(runs_root, variant_name)
        os.makedirs(variant_dir, exist_ok=True)

        # Reuse your existing LOCO runner (it will emit summary_loco.csv under variant_dir)
        run_loco_grid(
            root_dir=root_dir,
            ds=ds,
            base_cfg=cfg,
            build_system_fn=build_system_fn,
            fit_fn=fit_fn,
            epochs=epochs,
            batch_size=batch_size,
            num_workers=num_workers,
            lambda_ssim=lambda_ssim,
            early_stop_patience=early_stop_patience,
            runs_root=variant_dir,
            folds=folds_to_use  # None -> derive from dataset/cities
        )

        # Pull the per-variant LOCO summary and tag with variant
        per_variant_summary = os.path.join(variant_dir, "summary_loco.csv")
        if os.path.exists(per_variant_summary):
            df = pd.read_csv(per_variant_summary)
            df.insert(0, "variant", variant_name)
            combined_rows.append(df)
        else:
            print(f"Warning: {per_variant_summary} missing; skipping aggregation.")

    # Write combined summary across variants
    if combined_rows:
        combo = pd.concat(combined_rows, ignore_index=True)
        out_path = os.path.join(runs_root, "summary_ablation_loco.csv")
        combo.to_csv(out_path, index=False)
        print(f"\n✅ Combined ablation summary saved: {out_path}")
    else:
        print("\nNo variant summaries found; nothing to aggregate.")


In [80]:
if __name__ == '__main__':
    import torch
    # --- 1. Configuration ---
    EPOCHS = 25
    BATCH_SIZE = 32
    NUM_WORKERS = 0
    OUTPUT_DIR = "./runs"
    FOLDS_TO_RUN = None

    DATA_DIR = "./"
    ROOT_DIR = DATA_DIR



    # --- 2. Seed and Load Data ---
    seed_everything(42)
    print("Loading UHI data...")
    try:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        city_data_cpu = load_city_packs(DATA_DIR)
        OUT_BANDS = city_data_cpu[next(iter(city_data_cpu))]['uhii'].shape[1]
        base_config = SystemConfig(
            lr=3e-4,
            weight_decay=1e-3,
            grad_clip_norm=0.5,
            warmup_steps=0,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            use_lc=True, use_lst=True, use_pm=True,
            lc_encoder_type='frac',
            tw=3,
            ch_lc=64, ch_lst=64, ch_pm=32,
            num_lc_classes=36, lc_emb_dim=8,
            use_stream_attn=True,
            use_unet_attn=True,
            depth=4,
            base_unet_width=128,
            out_feat_width=256,
            bottleneck_dilations=(1, 1),
            use_transpose_up=True,
            dropout_p=0.2,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            amp=True,
            total_steps=None,
        )
        lst_stats = compute_lst_normalization_stats(city_data_cpu, sorted(city_data_cpu.keys()))
        base_config = replace(base_config, out_bands=OUT_BANDS, pin_memory=(device.type == 'cpu'))
        cities = sorted(city_data_cpu.keys())
        if device.type != 'cpu':
            city_data = move_city_packs_to_device(city_data_cpu, device, non_blocking=True)
        else:
            city_data = city_data_cpu
        print(f"✅ Data loaded for {len(cities)} cities (device: {device.type.upper()}).")
    except Exception as e:
        print(f"❌ Failed to load data from {DATA_DIR}.")
        raise e

    ds = UHIMemoryEfficientDataset(cities, city_data, train=True, tw=3, use_precomputed_lc=True, lst_stats=lst_stats)

    # --- 3. Launch Ablation + LOCO ---
    print("\n🚀 Starting full ablation + LOCO grid...")
    run_all_ablation_loco(
        root_dir=DATA_DIR,
        ds=ds,
        cities=cities,
        base_cfg=base_config,
        build_system_fn=build_system,
        fit_fn=fit,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        lambda_ssim=0.2,
        early_stop_patience=10,
        runs_root=OUTPUT_DIR,
        folds=FOLDS_TO_RUN
    )

    print(f"\n🎉 Finished. Results in '{OUTPUT_DIR}'.")

Loading UHI data...
✅ Data loaded for 6 cities.
LST normalization: mean=21.403, std=12.354

🚀 Starting full ablation + LOCO grid...

##### VARIANT: plain_d3_cbam #####
Starting LOCO evaluation for 6 folds: ['Bengaluru', 'Chennai', 'Delhi', 'Kolkata', 'Mumbai', 'Pune']

========================= FOLD: Holding out BENGALURU =========================
Training on 1020 samples, validating on 60...


e:\APUHI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
e:\APUHI\.venv\Lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Debug: MAE=4.569, SSIM=0.001, Total Loss=4.769
Debug: MAE=4.682, SSIM=0.000, Total Loss=4.882
Debug: MAE=4.501, SSIM=0.000, Total Loss=4.701
Debug: MAE=4.246, SSIM=0.000, Total Loss=4.446
Debug: MAE=4.276, SSIM=0.001, Total Loss=4.476
Debug: MAE=4.116, SSIM=0.001, Total Loss=4.316
Debug: MAE=3.855, SSIM=0.001, Total Loss=4.055
Debug: MAE=3.844, SSIM=0.001, Total Loss=4.044
Debug: MAE=3.677, SSIM=0.001, Total Loss=3.877
Debug: MAE=3.262, SSIM=0.002, Total Loss=3.461
Debug: MAE=3.061, SSIM=0.001, Total Loss=3.261
Debug: MAE=2.759, SSIM=0.002, Total Loss=2.959
Debug: MAE=2.601, SSIM=0.002, Total Loss=2.800
Debug: MAE=2.341, SSIM=0.001, Total Loss=2.540
Debug: MAE=1.989, SSIM=0.001, Total Loss=2.189
Debug: MAE=1.817, SSIM=0.003, Total Loss=2.016
Debug: MAE=1.558, SSIM=0.002, Total Loss=1.758
Debug: MAE=1.395, SSIM=0.003, Total Loss=1.595
Debug: MAE=1.497, SSIM=0.003, Total Loss=1.697
Debug: MAE=1.317, SSIM=0.007, Total Loss=1.516
Debug: MAE=1.292, SSIM=0.007, Total Loss=1.490
Debug: MAE=1.

Evaluating test set:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Evaluation complete. Results saved to: ./runs\plain_d3_cbam\LOCO_Bengaluru
CBAM attention maps saved to: ./runs\plain_d3_cbam\LOCO_Bengaluru\interpretability\cbam
✅ U-Net attention gate masks saved to ./runs\plain_d3_cbam\LOCO_Bengaluru\interpretability\gates
Results for fold Bengaluru: MAE=0.5828, R2=0.3064

========================= FOLD: Holding out CHENNAI =========================
Training on 1020 samples, validating on 60...
Debug: MAE=5.025, SSIM=0.000, Total Loss=5.225
Debug: MAE=4.962, SSIM=0.001, Total Loss=5.161
Debug: MAE=4.938, SSIM=0.000, Total Loss=5.138
Debug: MAE=4.896, SSIM=0.001, Total Loss=5.096
Debug: MAE=4.318, SSIM=0.000, Total Loss=4.518
Debug: MAE=4.162, SSIM=0.001, Total Loss=4.362
Debug: MAE=4.151, SSIM=0.001, Total Loss=4.351
Debug: MAE=3.780, SSIM=0.000, Total Loss=3.980
Debug: MAE=3.365, SSIM=0.001, Total Loss=3.565
Debug: MAE=3.186, SSIM=0.001, Total Loss=3.385
Debug: MAE=3.041, SSIM=0.002, Total Loss=3.240
Debug: MAE=2.769, SSIM=0.001, Total Loss=2.969

...

e:\APUHI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
e:\APUHI\.venv\Lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Debug: MAE=4.709, SSIM=0.001, Total Loss=4.909
Debug: MAE=4.992, SSIM=0.001, Total Loss=5.192
Debug: MAE=4.839, SSIM=0.001, Total Loss=5.039
Debug: MAE=4.650, SSIM=0.000, Total Loss=4.850
Debug: MAE=4.468, SSIM=0.001, Total Loss=4.668
Debug: MAE=3.943, SSIM=0.000, Total Loss=4.143
Debug: MAE=4.002, SSIM=0.001, Total Loss=4.201
Debug: MAE=3.439, SSIM=0.000, Total Loss=3.639
Debug: MAE=3.372, SSIM=0.001, Total Loss=3.571
Debug: MAE=2.846, SSIM=0.001, Total Loss=3.046
Debug: MAE=2.413, SSIM=0.001, Total Loss=2.613
Debug: MAE=2.546, SSIM=0.001, Total Loss=2.746
Debug: MAE=2.162, SSIM=0.001, Total Loss=2.361
Debug: MAE=2.017, SSIM=0.002, Total Loss=2.217
Debug: MAE=1.764, SSIM=0.004, Total Loss=1.964
Debug: MAE=1.456, SSIM=0.005, Total Loss=1.655
Debug: MAE=1.263, SSIM=0.005, Total Loss=1.462
Debug: MAE=1.302, SSIM=0.006, Total Loss=1.501
Debug: MAE=1.166, SSIM=0.005, Total Loss=1.365
Debug: MAE=1.256, SSIM=0.007, Total Loss=1.454
Debug: MAE=1.191, SSIM=0.007, Total Loss=1.389
Debug: MAE=1.

Evaluating test set:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Evaluation complete. Results saved to: ./runs\plain_d4_cbam\LOCO_Bengaluru
CBAM attention maps saved to: ./runs\plain_d4_cbam\LOCO_Bengaluru\interpretability\cbam
✅ U-Net attention gate masks saved to ./runs\plain_d4_cbam\LOCO_Bengaluru\interpretability\gates
Results for fold Bengaluru: MAE=0.6059, R2=-1.1738

========================= FOLD: Holding out CHENNAI =========================
Training on 1020 samples, validating on 60...
Debug: MAE=5.392, SSIM=0.001, Total Loss=5.592
Debug: MAE=5.260, SSIM=0.001, Total Loss=5.460
Debug: MAE=5.076, SSIM=0.000, Total Loss=5.276
Debug: MAE=4.854, SSIM=0.000, Total Loss=5.054
Debug: MAE=4.789, SSIM=0.001, Total Loss=4.988
Debug: MAE=4.440, SSIM=0.000, Total Loss=4.640
Debug: MAE=4.405, SSIM=0.001, Total Loss=4.605
Debug: MAE=4.314, SSIM=0.001, Total Loss=4.514
Debug: MAE=3.762, SSIM=0.001, Total Loss=3.961
Debug: MAE=3.705, SSIM=0.001, Total Loss=3.905
Debug: MAE=3.256, SSIM=0.001, Total Loss=3.455
Debug: MAE=3.368, SSIM=0.001, Total Loss=3.56

KeyboardInterrupt: 

In [ ]:
# import os
# import json
# import torch
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from tqdm import tqdm
# from scipy.stats import wilcoxon

# # Important: Make sure your model definitions (e.g., UHINet) and dataset
# # (UHIMemoryEfficientDataset) are importable from other files.
# # from model import UHINet, SystemConfig, build_system
# # from dataset import UHIMemoryEfficientDataset

# # ===================================================================
# # SECTION 1: ATTENTION-LAND COVER ANALYSIS
# # ===================================================================
# # This is the key function for generating publishable insights.

# @torch.no_grad()
# def analyze_attention_vs_land_cover(model, loader, device, save_dir):
#     """
#     Analyzes the model's spatial attention weights against land cover classes.
#     This provides quantitative evidence of what the model is "looking at".
#     """
#     os.makedirs(save_dir, exist_ok=True)
#     model.eval()

#     # We need to capture the spatial attention map from the LCEncoder's final CBAM block
#     attention_maps = []
#     lc_data = []

#     # Hook to capture the input to the final CBAM's spatial attention part
#     captured_features = {}
#     def hook_fn(module, input, output):
#         captured_features['spatial_input'] = output.detach()

#     # Find the target module to hook
#     # NOTE: Adjust the name if your model structure is different
#     target_module_name = "lc_encoder.block2.cbam"
#     hook_handle = None
#     for name, module in model.named_modules():
#         if name == target_module_name:
#             hook_handle = module.register_forward_hook(hook_fn)
#             print(f"✅ Hook registered on '{name}'")
#             break

#     if not hook_handle:
#         print(f"⚠️ Could not find target module '{target_module_name}' to hook for attention analysis.")
#         return

#     print("--- Running model to capture attention maps and LC data ---")
#     for batch in tqdm(loader, desc="Analyzing Attention"):
#         lc = batch["lc"].to(device).long()
#         _ = model(lc=lc, pm=batch["pm"].to(device), lst=batch["lst"].to(device)) # Dummy run to trigger hook

#         if 'spatial_input' in captured_features:
#             # Re-compute the spatial attention map from the captured feature
#             cbam_module = dict(model.named_modules())[target_module_name]
#             spatial_map = cbam_module.sigmoid(cbam_module.spatial_conv(torch.cat([
#                 torch.mean(captured_features['spatial_input'], dim=1, keepdim=True),
#                 torch.max(captured_features['spatial_input'], dim=1, keepdim=True)[0]
#             ], dim=1)))

#             # Upsample attention map to match the LC grid size for comparison
#             upsampled_attention = F.interpolate(spatial_map, size=lc.shape[-2:], mode='bilinear', align_corners=False)
#             attention_maps.append(upsampled_attention.cpu())
#             lc_data.append(lc.cpu())

#     hook_handle.remove() # Clean up the hook

#     if not attention_maps:
#         print("No attention maps were captured. Aborting analysis.")
#         return

#     attention_tensor = torch.cat(attention_maps, dim=0).squeeze(1) # Shape: (N, H, W)
#     lc_tensor = torch.cat(lc_data, dim=0).squeeze(1) # Shape: (N, H, W)

#     # --- Calculate Average Attention per LC Class ---
#     num_lc_classes = 36 # As defined in your model
#     avg_attention_per_class = []
#     for i in tqdm(range(num_lc_classes), desc="Calculating avg attention"):
#         mask = (lc_tensor == i)
#         if mask.sum() > 0:
#             avg_attention = attention_tensor[mask].mean().item()
#             avg_attention_per_class.append(avg_attention)
#         else:
#             avg_attention_per_class.append(0)

#     # --- Save and Plot Results ---
#     # This is your key quantitative result for the paper
#     lc_class_names = [f"Class_{i}" for i in range(num_lc_classes)] # Replace with actual names if you have them
#     df = pd.DataFrame({'Land_Cover_Class': lc_class_names, 'Average_Attention': avg_attention_per_class})
#     df.to_csv(os.path.join(save_dir, "attention_by_lc_class.csv"), index=False)
#     print(f"📄 Attention-by-LC analysis saved to {save_dir}")

#     # Plot for visualization
#     plt.figure(figsize=(12, 7))
#     df_sorted = df.sort_values('Average_Attention', ascending=False)
#     plt.bar(df_sorted['Land_Cover_Class'], df_sorted['Average_Attention'])
#     plt.xticks(rotation=90)
#     plt.ylabel("Average Spatial Attention Weight")
#     plt.title("Model Attention by Land Cover Class")
#     plt.tight_layout()
#     plt.savefig(os.path.join(save_dir, "attention_by_lc_class.png"), dpi=150)
#     plt.close()

# # ===================================================================
# # SECTION 2: STATISTICAL SIGNIFICANCE TESTING
# # ===================================================================

# def run_statistical_tests(runs_root: str, main_model: str, baselines: List[str]):
#     """
#     Compares model performance across LOCO folds using a paired statistical test.

#     Args:
#         runs_root: The root directory containing all experiment runs.
#         main_model: The name of your main model's run directory.
#         baselines: A list of names for baseline model directories to compare against.
#     """
#     print("\n--- Running Statistical Significance Tests ---")
#     try:
#         main_df_path = os.path.join(runs_root, main_model, "test_per_city.csv")
#         main_df = pd.read_csv(main_df_path).set_index("city")
#     except FileNotFoundError:
#         print(f"⚠️ Main model results not found at {main_df_path}. Cannot perform tests.")
#         return

#     results = []
#     for baseline_name in baselines:
#         try:
#             baseline_df_path = os.path.join(runs_root, baseline_name, "test_per_city.csv")
#             baseline_df = pd.read_csv(baseline_df_path).set_index("city")

#             # Align dataframes by city
#             merged = main_df.join(baseline_df, lsuffix='_main', rsuffix='_baseline', how='inner')
#             if len(merged) < 5: # Need enough pairs for a meaningful test
#                 print(f"Skipping {baseline_name}: Not enough common cities found.")
#                 continue

#             stat, p_value = wilcoxon(merged['mae_main'], merged['mae_baseline'])
#             results.append({
#                 "comparison": f"{main_model}_vs_{baseline_name}",
#                 "p_value_mae": p_value,
#                 "is_significant_at_0.05": p_value < 0.05
#             })
#         except FileNotFoundError:
#             print(f"⚠️ Baseline model results not found for '{baseline_name}'. Skipping.")

#     if results:
#         results_df = pd.DataFrame(results)
#         results_path = os.path.join(runs_root, "statistical_test_summary.csv")
#         results_df.to_csv(results_path, index=False)
#         print("📄 Statistical test results saved to 'statistical_test_summary.csv'")
#         print(results_df)

# if __name__ == '__main__':
#     # This is an example of how you would run these analysis scripts after training.

#     # You would first load your best model and test dataloader
#     # For demonstration, we will assume they are loaded.
#     # ------------------------------------------------------------------
#     # best_model = ...
#     # test_loader = ...
#     # device = torch.device("cuda")
#     # analyze_attention_vs_land_cover(best_model, test_loader, device, save_dir="./runs/main_model_run/analysis")
#     # ------------------------------------------------------------------

#     # After running all your ablation experiments, you would run the statistical tests
#     run_statistical_tests(
#         runs_root="./runs",
#         main_model="att_d4_cbam_main", # Name of your main model's experiment folder
#         baselines=["plain_d3_cbam", "plain_d4_cbam"] # Names of baseline folders
#     )

In [ ]:

# =======================
# 🩹 Patch Cell — UHINet full pipeline fixes
# =======================
# This cell overrides and augments earlier functions to implement:
# 1) LR scheduler created inside `fit()` (no total_steps requirement in build_system)
# 2) LC precompute mode tied to actual availability; per-city handling
# 3) Consistent geometric augs across ALL streams (incl. 30 m LC)
# 4) Precompute LC features using lc_annual.pt + month_to_year.pt mapping
# 5) Earth Engine reprojection/masking fixes (no sign-based mask)
# 6) Normalization computed from TRAIN ONLY
# 7) Complete CBAM interpretability dumper
# 8) Quiet compute_loss (no spam prints)

import os, json, math, time, csv
from dataclasses import asdict
from typing import Dict, List, Tuple, Optional, Union
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import LambdaLR
except Exception:
    pass

def make_warmup_cosine_scheduler(optimizer: torch.optim.Optimizer,
                                 total_steps: int,
                                 warmup_steps: int = 0):
    warmup_steps = max(0, warmup_steps)
    total_steps = max(1, total_steps)
    def lr_lambda(step: int):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)

def exclude_from_weight_decay(n: str, p: nn.Parameter) -> bool:
    if p.ndim == 1: return True
    if n.endswith(".bias"): return True
    norm_keywords = ["norm", "gn", "bn", "ln", "in", "running_mean", "running_var"]
    if any(k in n.lower() for k in norm_keywords): return True
    if "embedding" in n.lower(): return True
    return False

def make_optimizer(model: nn.Module, lr: float = 3e-4, weight_decay: float = 1e-2) -> torch.optim.Optimizer:
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (no_decay if exclude_from_weight_decay(n, p) else decay).append(p)
    return torch.optim.AdamW([
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ], lr=lr, betas=(0.9, 0.999))

def build_encoders(cfg, lc_precomputed: bool | None = None):
    if cfg.lc_encoder_type == "cbam":
        lc_enc = LCEncoder(num_classes=cfg.num_lc_classes, emb_dim=cfg.lc_emb_dim)
    elif cfg.lc_encoder_type == "frac":
        lc_enc = LCFracEncoder(num_classes=cfg.num_lc_classes, out_ch=cfg.ch_lc, precomputed=bool(lc_precomputed))
    else:
        raise ValueError(f"Unknown lc_encoder_type: {cfg.lc_encoder_type}")
    pm_enc  = PMEncoder(tw=cfg.tw)
    lst_enc = LSTEncoder(tw=cfg.tw)
    return lc_enc, pm_enc, lst_enc

def detect_lc_precompute_availability(root_dir: str, cities: List[str]) -> tuple[Dict[str, bool], bool]:
    avail = {c: os.path.exists(os.path.join(root_dir, c, "lc_encoded.pt")) for c in cities}
    return avail, (all(avail.values()) if avail else False)

def build_system(root_dir: str, cfg, cities_priority: Optional[List[str]] = None, device: Optional[torch.device] = None):
    out_bands = cfg.out_bands or read_bands_from_meta(root_dir, cities_priority)
    cities = [d for d in sorted(os.listdir(root_dir)) if os.path.isdir(os.path.join(root_dir, d))]
    per_city_avail, all_avail = detect_lc_precompute_availability(root_dir, cities)
    lc_enc, pm_enc, lst_enc = build_encoders(cfg, lc_precomputed=all_avail)
    model = build_uhi_model(cfg, out_bands, lc_enc, lst_enc, pm_enc)
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = make_optimizer(model, lr=cfg.lr, weight_decay=cfg.weight_decay)
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler(
            device="cuda" if torch.cuda.is_available() else "cpu",
            enabled=cfg.amp,
        )
    else:
        scaler = GradScaler(enabled=cfg.amp)
    return {"model": model, "optimizer": optimizer, "scheduler": None, "scaler": scaler,
            "cfg": asdict(cfg), "param_counts": count_params(model), "device": device,
            "lc_precompute_available": per_city_avail}

def compute_loss(yhat: torch.Tensor, y: torch.Tensor, m: torch.Tensor, lambda_ssim: float = 0.0):
    assert yhat.shape[1] == y.shape[1] == m.shape[1]
    mae = (torch.abs(yhat - y) * m).sum() / m.sum().clamp(min=1e-6)
    if lambda_ssim > 0:
        ssim_val = masked_ssim(yhat, y, m)
        return mae + lambda_ssim * (1.0 - ssim_val), {"mae": float(mae), "ssim": float(ssim_val)}
    return mae, {"mae": float(mae)}

def fit(system, train_loader, val_loader, out_dir: str, epochs: int = 20, lambda_ssim: float = 0.2, early_stop_patience: int = 10):
    os.makedirs(out_dir, exist_ok=True)
    steps_per_epoch = max(1, len(train_loader))
    total_steps = epochs * steps_per_epoch
    if system.get("scheduler", None) is None:
        warmup_steps = system["cfg"].get("warmup_steps", 0)
        system["scheduler"] = make_warmup_cosine_scheduler(system["optimizer"], total_steps, warmup_steps)
    model, optimizer = system["model"], system["optimizer"]
    scheduler, scaler = system["scheduler"], system["scaler"]
    device = system["device"]
    if "cfg" in system:
        with open(os.path.join(out_dir, "config.json"), "w") as f: json.dump(system["cfg"], f, indent=2)
    best_val_mae = float("inf"); best_path = os.path.join(out_dir, "best.pt"); last_path = os.path.join(out_dir, "last.pt")
    history = {"train_loss": [], "train_mae": [], "val_loss": [], "val_mae": [], "val_rmse": [], "val_ssim": [], "lr": []}
    patience = 0
    for epoch in range(1, epochs+1):
        t0 = time.time()
        train_stats = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, epoch=epoch-1, steps_per_epoch=steps_per_epoch,
                                      device=device, lambda_ssim=lambda_ssim, grad_clip_norm=system["cfg"].get("grad_clip_norm", 0.5),
                                      warmup_steps=system["cfg"].get("warmup_steps", 0))
        val_stats = evaluate_epoch(model, val_loader, device, lambda_ssim=0.0) if val_loader is not None else \
                    {"loss": float('nan'), "mae": float('nan'), "rmse": float('nan'), "ssim": float('nan')}
        history["train_loss"].append(train_stats["loss"]); history["train_mae"].append(train_stats["mae"])
        history["val_loss"].append(val_stats["loss"]); history["val_mae"].append(val_stats["mae"])
        history["val_rmse"].append(val_stats["rmse"]); history["val_ssim"].append(val_stats["ssim"])
        history["lr"].append(optimizer.param_groups[0]["lr"])
        dt = time.time() - t0
        print(f"Epoch {epoch:03d} | train: loss {train_stats['loss']:.4f}, mae {train_stats['mae']:.4f} | "
              f"val: loss {val_stats['loss']:.4f}, mae {val_stats['mae']:.4f}, rmse {val_stats['rmse']:.4f}, ssim {val_stats['ssim']:.4f} | "
              f"lr {history['lr'][-1]:.2e} | {dt:.1f}s")
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict()}, last_path)
        if val_stats["mae"] < best_val_mae - 1e-6:
            best_val_mae = val_stats["mae"]; patience = 0
            torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict()}, best_path)
        else:
            patience += 1
        if patience >= early_stop_patience:
            print(f"Early stopping at epoch {epoch} (no val MAE improvement for {early_stop_patience} epochs)."); break
        _save_history_csv(history, os.path.join(out_dir, "history.csv"))
        _plot_curves(history, os.path.join(out_dir, "curves.png"), title="Loss/Metric Curves (train vs val)")
    _save_history_csv(history, os.path.join(out_dir, "history.csv"))
    _plot_curves(history, os.path.join(out_dir, "curves.png"), title="Loss/Metric Curves (train vs val)")
    return history, best_val_mae

def apply_shared_geometric_augs(sample: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    """Apply the same random flip/rotation to all spatial tensors in ``sample``."""
    do_vflip = torch.rand(1).item() > 0.5
    do_hflip = torch.rand(1).item() > 0.5
    do_rot90 = torch.rand(1).item() > 0.5
    for key in ["pm", "pm_mask", "lst", "lst_mask", "uhii", "uhii_mask", "lc"]:
        if key not in sample or not isinstance(sample[key], torch.Tensor):
            continue
        x = sample[key]
        if do_vflip:
            x = x.flip(-2)
        if do_hflip:
            x = x.flip(-1)
        if do_rot90:
            x = torch.rot90(x, 1, [-2, -1])
        sample[key] = x
    return sample


def _accumulate_tensor_stats(t: torch.Tensor, totals: Dict[str, float]):
    if not isinstance(t, torch.Tensor):
        return
    tensor = t.detach()
    if tensor.device.type != "cpu":
        tensor = tensor.to("cpu")
    flat = tensor.reshape(-1).double()
    totals["sum"] += flat.sum().item()
    totals["sum_sq"] += flat.pow(2).sum().item()
    totals["count"] += flat.numel()


def compute_lst_normalization_stats(data: Dict[str, dict], cities: Optional[List[str]] = None) -> Tuple[float, float]:
    totals = {"sum": 0.0, "sum_sq": 0.0, "count": 0}
    for city in (cities or list(data.keys())):
        pack = data.get(city, {})
        lst = pack.get("lst")
        if isinstance(lst, torch.Tensor):
            _accumulate_tensor_stats(lst, totals)
    if totals["count"] == 0:
        raise RuntimeError("No LST tensors found for normalization.")
    mean = totals["sum"] / totals["count"]
    var = max(totals["sum_sq"] / totals["count"] - mean * mean, 0.0)
    std = max(var ** 0.5, 1e-6)
    return float(mean), float(std)


def move_city_packs_to_device(city_data: Dict[str, dict], device: Union[torch.device, str], *,
                              non_blocking: bool = True, inplace: bool = False) -> Dict[str, dict]:
    target = torch.device(device)
    if inplace:
        dest = city_data
    else:
        dest = {}
    for city, pack in city_data.items():
        target_pack = pack if inplace else {}
        for key, value in pack.items():
            if isinstance(value, torch.Tensor):
                if value.device == target:
                    moved = value
                else:
                    moved = value.to(target, non_blocking=non_blocking)
                target_pack[key] = moved
            else:
                target_pack[key] = value
        if not inplace:
            dest[city] = target_pack
    return dest


class UHIMemoryEfficientDataset(torch.utils.data.Dataset):
    def __init__(self,
                 cities: List[str],
                 data: Dict[str, dict],
                 train: bool = True,
                 tw: int = 3,
                 use_precomputed_lc: bool = True,
                 lst_stats: Optional[Tuple[float, float]] = None,
                 target_device: Optional[Union[torch.device, str]] = None,
                 non_blocking: bool = True):
        self.cities = list(cities)
        self.data = data
        self.train = train
        self.tw = tw
        self.use_precomputed_lc = use_precomputed_lc
        self.non_blocking = non_blocking

        if target_device is not None:
            move_city_packs_to_device(self.data, target_device, non_blocking=non_blocking, inplace=True)

        self.device = self._infer_data_device()

        if lst_stats is not None:
            mean, std = lst_stats
        else:
            mean, std = compute_lst_normalization_stats(self.data, self.cities)
        self.lst_mean = float(mean)
        self.lst_std = max(float(std), 1e-6)

        self.lc_precomp_avail = {c: ("lc_encoded" in self.data[c]) for c in self.cities}

        self.indices = []
        for city in self.cities:
            uhii = self.data[city]["uhii"]
            self.indices.extend([(city, t) for t in range(uhii.shape[0])])

    def _infer_data_device(self) -> torch.device:
        for city in self.cities:
            pack = self.data[city]
            for key in ("pm25", "uhii", "lst"):
                tensor = pack.get(key)
                if isinstance(tensor, torch.Tensor):
                    return tensor.device
        return torch.device("cpu")

    def __len__(self) -> int:
        return len(self.indices)

    def _window_stack(self, x: torch.Tensor, t: int, per_step_ch: int) -> torch.Tensor:
        T, C, H, W = x.shape
        start = max(0, t - self.tw + 1)
        pad_left = self.tw - (t - start + 1)
        chunks = [x[start:t+1]]
        if pad_left > 0:
            chunks.insert(0, x[0:1].expand(pad_left, C, H, W))
        win = torch.cat(chunks, dim=0)
        return win.permute(1, 0, 2, 3).reshape(C * self.tw, H, W)

    def __getitem__(self, idx: int) -> Dict[str, Union[torch.Tensor, str]]:
        city, t = self.indices[idx]
        pack = self.data[city]

        pm = self._window_stack(pack['pm25'], t, 1)
        pm_mask = self._window_stack(pack['pm25_mask'], t, 1)
        lst = self._window_stack(pack['lst'], t, 2)
        lst_mask = self._window_stack(pack['lst_mask'], t, 2)

        pm = torch.clamp(pm, 0, 150.0) / 150.0
        lst = (lst - self.lst_mean) / self.lst_std

        if self.use_precomputed_lc and self.lc_precomp_avail.get(city, False) and ('lc_encoded' in pack):
            lc = pack['lc_encoded'][t]
        else:
            lc_src = pack.get('lc')
            if lc_src is None:
                raise KeyError(f"City {city} is missing 'lc' data")
            lc_t = lc_src[t]
            if isinstance(lc_t, torch.Tensor) and lc_t.shape[0] == 1 and lc_t.shape[-1] >= 1000:
                lc = lc_t.long()
            else:
                lc = lc_t.float() if isinstance(lc_t, torch.Tensor) else lc_t

        uhii = pack['uhii'][t]
        uhii_mask = pack['uhii_mask'][t]

        sample = {
            "pm": pm.float(),
            "pm_mask": pm_mask.float(),
            "lst": lst.float(),
            "lst_mask": lst_mask.float(),
            "lc": lc,
            "uhii": uhii.float(),
            "uhii_mask": uhii_mask.float(),
            "city": city,
            "t": t,
        }

        if self.train:
            sample = apply_shared_geometric_augs(sample)

        return sample


def compute_and_set_lst_stats_from_indices(ds, indices: List[int]):
    totals = {"sum": 0.0, "sum_sq": 0.0, "count": 0}
    for i in indices:
        city, t = ds.indices[i]
        x = ds.data[city]['lst'][t]
        if isinstance(x, torch.Tensor):
            _accumulate_tensor_stats(x, totals)
    if totals["count"] == 0:
        raise RuntimeError("No indices provided for normalization stats.")
    mean = totals["sum"] / totals["count"]
    var = max(totals["sum_sq"] / totals["count"] - mean * mean, 0.0)
    std = max(var ** 0.5, 1e-6)
    ds.lst_mean = float(mean)
    ds.lst_std = float(std)
    print(f"[norm] LST stats set from train only: mean={mean:.4f}, std={std:.4f}")
    return float(mean), float(std)

try:
    import ee, geemap
    PIXEL_SIZE_1KM = 1000; PIXEL_SIZE_30M = 30
    def to1km(img: 'ee.Image', region: 'ee.Geometry') -> 'ee.Image':
        return img.reproject(img.projection().atScale(PIXEL_SIZE_1KM)).clip(region)
    def fetch_uhii_month(asset: str, year: int, month: int, region: 'ee.Geometry') -> 'ee.Image':
        coll = (ee.ImageCollection(asset).filter(ee.Filter.calendarRange(year, year, 'year'))
                .filter(ee.Filter.calendarRange(month, month, 'month')).filterBounds(region))
        mean_img = coll.mean(); count_img = coll.count().reduce(ee.Reducer.max())
        return to1km(mean_img.updateMask(count_img.gt(0)), region)
    def fetch_pm25_month(year: int, month: int, region: 'ee.Geometry') -> 'ee.Image':
        start = ee.Date.fromYMD(year, month, 1); end = start.advance(1, 'month')
        coll = (ee.ImageCollection("projects/sat-io/open-datasets/GLOBAL-SATELLITE-PM25/MONTHLY")
                .filterDate(start, end).filterBounds(region))
        mean_img = coll.mean(); count_img = coll.count().reduce(ee.Reducer.max())
        return to1km(mean_img.updateMask(count_img.gt(0)).rename(["PM25"]), region)
except Exception as _ee_err:
    pass

@torch.no_grad()
def precompute_lc_features(root_dir: str, cities: List[str], cfg, encoder_type: str = "frac",
                           overwrite: bool = False, device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")):
    if encoder_type == "frac":
        lc_enc = LCFracEncoder(num_classes=cfg.num_lc_classes, out_ch=cfg.ch_lc, precomputed=False)
    elif encoder_type == "cbam":
        lc_enc = LCEncoder(num_classes=cfg.num_lc_classes, emb_dim=cfg.lc_emb_dim)
    else:
        raise ValueError("encoder_type must be 'frac' or 'cbam'")
    lc_enc = lc_enc.to(device).eval()
    for city in cities:
        city_dir = os.path.join(root_dir, city)
        if not os.path.isdir(city_dir): continue
        out_p = os.path.join(city_dir, "lc_encoded.pt")
        if os.path.exists(out_p) and not overwrite:
            print(f"[precompute] Skip {city}: lc_encoded.pt exists."); continue
        lc_ann_p = os.path.join(city_dir, "lc_annual.pt"); m2y_p = os.path.join(city_dir, "month_to_year.pt"); uh_p = os.path.join(city_dir, "uhii.pt")
        if not (os.path.exists(lc_ann_p) and os.path.exists(m2y_p) and os.path.exists(uh_p)):
            print(f"[precompute] Missing inputs for {city} (need lc_annual.pt, month_to_year.pt, uhii.pt)."); continue
        lc_annual = torch.load(lc_ann_p, map_location="cpu").to(device)   # (Y,1,1333,1333)
        month_to_year = torch.load(m2y_p, map_location="cpu").long()       # (T,)
        T = torch.load(uh_p, map_location="cpu").shape[0]
        feats_year = [];  [feats_year.append(lc_enc(lc_annual[y:y+1]).detach().cpu()) for y in range(lc_annual.shape[0])]
        feats_year = torch.cat(feats_year, dim=0)                          # (Y,64,40,40)
        feats_monthly = feats_year[month_to_year[:T]].contiguous()         # (T,64,40,40)
        torch.save(feats_monthly, out_p)
        meta_p = os.path.join(city_dir, "meta.json")
        if os.path.exists(meta_p):
            with open(meta_p, "r") as f: meta = json.load(f)
            meta.setdefault("files", {})["lc_encoded"] = "lc_encoded.pt"
            meta["lc_encoded_shape"] = [int(s) for s in feats_monthly.shape]
            with open(meta_p, "w") as f: json.dump(meta, f, indent=2)
        print(f"[precompute] {city}: saved {tuple(feats_monthly.shape)} to lc_encoded.pt")

def _cbam_channel_att(module, x):
    B, C, H, W = x.shape
    avg = module.avg_pool(x).view(B, C); mx  = module.max_pool(x).view(B, C)
    return module.sigmoid(module.mlp(avg) + module.mlp(mx)).view(B, C, 1, 1)

def _cbam_spatial_att(module, x_after_channel):
    avg_sp = torch.mean(x_after_channel, dim=1, keepdim=True)
    max_sp, _ = torch.max(x_after_channel, dim=1, keepdim=True)
    return torch.sigmoid(module.spatial_conv(torch.cat([avg_sp, max_sp], dim=1)))

@torch.no_grad()
def dump_lc_cbam_maps(model, loader, device, save_dir: str, max_samples: int = 6):
    os.makedirs(save_dir, exist_ok=True)
    targets = [(name, mod) for name, mod in model.named_modules()
               if name.endswith("lc_encoder.block1.cbam") or name.endswith("lc_encoder.block2.cbam")]
    if not targets:
        print("[interpret] No LC CBAM modules found."); return
    saved = 0; sample_id = 0; channel_rows = []
    for batch in loader:
        if saved >= max_samples: break
        pm, pm_m = batch["pm"].to(device), batch["pm_mask"].to(device)
        lst, lst_m = batch["lst"].to(device), batch["lst_mask"].to(device)
        lc = batch["lc"].to(device).long()
        cities = batch["city"]
        captured_inputs = {}
        def make_pre(name):
            def _pre_hook(m, inp): captured_inputs[name] = inp[0].detach()
            return _pre_hook
        handles = [mod.register_forward_pre_hook(make_pre(name)) for name, mod in targets]
        _ = model(pm=pm, pm_mask=pm_m, lst=lst, lst_mask=lst_m, lc=lc)
        for h in handles: h.remove()
        for name, cbam in targets:
            if name not in captured_inputs: continue
            x_in = captured_inputs[name].to(device)
            ca = _cbam_channel_att(cbam, x_in).cpu().squeeze(-1).squeeze(-1)
            x_after_ca = x_in * ca.unsqueeze(-1).unsqueeze(-1).to(device)
            sa = _cbam_spatial_att(cbam, x_after_ca).cpu()
            B = x_in.shape[0]
            import matplotlib.pyplot as plt, numpy as np
            for b in range(B):
                city = cities[b]
                plt.figure(figsize=(4,4)); plt.imshow(sa[b,0], interpolation="nearest"); plt.title(f"{name} spatial — {city}")
                plt.axis("off"); plt.tight_layout()
                plt.savefig(os.path.join(save_dir, f"cbam_spatial_{sample_id}_{name.replace('.', '_')}_{city}.png"), dpi=140); plt.close()
                plt.figure(figsize=(6,2.6)); plt.bar(np.arange(ca.shape[1]), ca[b].numpy()); plt.title(f"{name} channel — {city}")
                plt.tight_layout(); plt.savefig(os.path.join(save_dir, f"cbam_channel_{sample_id}_{name.replace('.', '_')}_{city}.png"), dpi=140); plt.close()
                channel_rows.append({"sample_id": int(sample_id), "module": name, "city": city, "channel_att": ca[b].double().numpy().tolist()})
                sample_id += 1; saved += 1
                if saved >= max_samples: break
            if saved >= max_samples: break
    if channel_rows:
        with open(os.path.join(save_dir, "cbam_channel_att.json"), "w") as f: json.dump(channel_rows, f, indent=2)
    print(f"[interpret] Saved CBAM maps to {save_dir}")
